# 📱🎛️🐨 Strata Studio (HTML5 ถาวร • 66 โมเดล • ระบบรีพายข้อความ Pro ถามต่อ/วาดภาพ/คลุมข้อความย่อย • Auto โหนด 4–7)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1PAarnmBZKYu8WQZct80X_J248Ne4fcS-)

- **เซลล์ที่ 1:** เตรียมสภาพแวดล้อม + โหลดแคตตาล็อกครบ **66 โมเดล (54 LLM + 6 Image GGUF + 6 Uncensored LoRA)** + แสดงตาราง **`strata diagnose`**
- **เซลล์ที่ 2:** ระบบ **↩️ รีพายข้อความ Pro** (รองรับลากคลุมเฉพาะประโยค คลิกกล่องอ้างอิงย้อนดูต้นทาง ปุ่มด่วนถามต่อ/วาดภาพ/แต่งโหนด/แก้ไข/คัดลอก/ลบรายข้อความ และคีย์ลัด `Esc` / `Ctrl+Enter`)
- **เซลล์ที่ 3:** **ComfyUI GGUF Node Workflow Studio** รับค่าซิงก์สดจากข้อความที่พิมพ์หรือรีพายในเซลล์ที่ 2 เข้าสู่ **Node 4–5** และ **Node 6–7** ทันที
- **เซลล์ที่ 1 ใช้บน 🐨 Kaggle:** ต้องเปิด **Settings → Internet → On** (หรือแนบ repo นี้เป็น **Dataset** แล้วไม่ต้องใช้เน็ต) — ตัวเซลล์จะหา *project root* เองอัตโนมัติ รองรับทั้ง `strata/` ที่รูท และโฟลเดอร์ย่อยอย่าง `multiple-projects/strata/`
- **repo เริ่มต้น:** `PingPhantom/Jupyter-Notebooks` กำหนดเองได้ที่ช่อง `REPO_SLUG`


In [1]:
# @title 1) เตรียมสภาพแวดล้อม + โหลดแคตตาล็อก 66 โมเดล + แสดงผล `strata diagnose` { display-mode: "form" }
REPO_SLUG = ""  # @param {type:"string"}
DEFAULT_SLUG = "PingPhantom/Jupyter-Notebooks"
REPO_BRANCH = ""  # @param {type:"string"}
AUTO_UPDATE = True  # @param {type:"boolean"}
USE_NIX = "auto"  # @param ["auto", "always", "never"]

NOTEBOOK_REV = "v52-fix-kaggle-project-root"
import glob, importlib, io, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
REQUIRED = ("strata/__init__.py", "pyproject.toml")
GIT_ENV = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
JUNK_DIRS = {".git", ".venv", "__pycache__", ".ipynb_checkpoints", "node_modules", "site-packages", ".mypy_cache", ".pytest_cache"}

def _is_kaggle() -> bool:
    return any(k.startswith("KAGGLE_") for k in os.environ) or Path("/kaggle/working").is_dir()

def _is_colab() -> bool:
    if _is_kaggle(): return False
    return "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ

def _ok(root) -> bool:
    """True เมื่อโฟลเดอร์นี้เป็น project root ของแพ็กเกจ strata จริง"""
    return all((Path(root) / f).is_file() for f in REQUIRED)

def _purge_modules() -> None:
    importlib.invalidate_caches()
    for name in [n for n in list(sys.modules) if n == "strata" or n.startswith("strata.")]:
        del sys.modules[name]

def _find_project_root(base) -> "Path | None":
    """หา project root จริงของ strata/ — รองรับทั้ง repo ที่วาง strata/ ไว้ที่รูท
    และ repo ที่ห่อโค้ดไว้ในโฟลเดอร์ย่อย ≤2 ชั้น (เช่น multiple-projects/strata/)
    คืน None ถ้าไม่พบ ทำให้เช็คได้ว่าควรโคลนใหม่หรือไม่"""
    base = Path(base)
    if _ok(base): return base
    if not base.is_dir(): return None
    found: list[Path] = []
    for pat in ("strata/__init__.py", "*/strata/__init__.py", "*/*/strata/__init__.py"):
        try:
            hits = sorted(base.glob(pat))
        except OSError:
            continue
        for init_py in hits:
            rel = init_py.relative_to(base).parts[:-2]
            if any(part in JUNK_DIRS or part.startswith(".") for part in rel): continue
            root = init_py.parent.parent
            if root not in found: found.append(root)
    found.sort(key=lambda r: (not _ok(r), len(r.relative_to(base).parts)))
    return found[0] if found else None

def _copy_project(src: Path, target: Path) -> Path:
    target.mkdir(parents=True, exist_ok=True)
    for item in sorted(src.iterdir()):
        if item.name in JUNK_DIRS or item.name.startswith("."): continue
        dest = target / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".git", ".venv"))
        else:
            shutil.copy2(item, dest)
    return target

def _sync_from_kaggle() -> "Path | None":
    """🐨 ถ้าแนบ repo นี้เป็น Kaggle Dataset (/kaggle/input/...) ใช้โค้ดจาก dataset ได้เลย
    โดยไม่ต้องเปิดอินเทอร์เน็ต — ตรวจทั้งกรณี dataset มี strata/ ที่รูท และซ้อนในโฟลเดอร์ย่อย"""
    inp = Path("/kaggle/input")
    if not inp.is_dir(): return None
    src = _find_project_root(inp)
    if src is None:
        print("ℹ️ ไม่พบแพ็กเกจ strata/ ใน /kaggle/input — จะโหลดจาก GitHub แทน", flush=True)
        return None
    try:
        target = _copy_project(src, Path("/kaggle/working/Colab"))
    except OSError as exc:
        print(f"⚠️ คัดลอกจาก Kaggle Dataset ไม่สำเร็จ ({exc}) — จะโหลดจาก GitHub แทน", flush=True)
        return None
    print(f"✅ ใช้แพ็กเกจ strata/ จาก Kaggle Dataset: {src}", flush=True)
    return _find_project_root(target) or target

def _git(*args, timeout: int = 180):
    return subprocess.run(["git", *args], capture_output=True, text=True, env=GIT_ENV, timeout=timeout, check=False)

def _git_fetch_ref(target: Path, url: str, rspec: str) -> bool:
    shutil.rmtree(target, ignore_errors=True)
    target.mkdir(parents=True, exist_ok=True)
    _git("-C", str(target), "init", "-q")
    _git("-C", str(target), "remote", "add", "origin", url)
    try:
        fetched = _git("-C", str(target), "fetch", "--depth", "1", "origin", rspec, timeout=300)
    except subprocess.TimeoutExpired:
        print(f"⏱️ git fetch {rspec} เกิน 300s — ข้าม", flush=True)
        return False
    if fetched.returncode != 0: return False
    _git("-C", str(target), "checkout", "-q", "-f", "FETCH_HEAD")
    return True

def _download_tarball(slug: str, branch: str, target: Path) -> bool:
    """Fallback ที่ไม่ต้องใช้ git (เผื่อ image ไม่มี git หรือ git ถูกบล็อก)"""
    refs = [f"refs/heads/{branch}"] if branch else ["refs/heads/main", "refs/heads/master", "HEAD"]
    for ref in refs:
        url = f"https://codeload.github.com/{slug}/tar.gz/{ref}"
        try:
            with urllib.request.urlopen(url, timeout=120) as resp:
                data = resp.read()
        except Exception as exc:
            print(f"⚠️ ดาวน์โหลด tarball ({ref}) ไม่ได้: {type(exc).__name__}", flush=True)
            continue
        shutil.rmtree(target, ignore_errors=True)
        target.mkdir(parents=True, exist_ok=True)
        with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tf:
            try:
                tf.extractall(target, filter="data")
            except TypeError:
                tf.extractall(target)
        top = [p for p in target.iterdir() if p.is_dir()]
        if len(top) == 1:  # codeload ห่อมาด้วย <repo>-<sha>/ เสมอ → ย้ายขึ้นระดับบน
            for item in list(top[0].iterdir()):
                shutil.move(str(item), str(target / item.name))
            shutil.rmtree(top[0], ignore_errors=True)
        return True
    return False

def _list_refs(url: str):
    """คืน ((heads, pulls), '') หรือ (None, stderr) เมื่อเข้า repo ไม่ได้"""
    try:
        probe = _git("ls-remote", url, timeout=30)
    except subprocess.TimeoutExpired:
        return None, "timeout 30s"
    if probe.returncode != 0:
        return None, (probe.stderr or probe.stdout or "").strip()
    heads, pulls = [], []
    for ln in probe.stdout.splitlines():
        pts = ln.strip().split()
        if len(pts) < 2: continue
        rf = pts[1]
        if rf.startswith("refs/pull/") and rf.endswith("/head") and rf[10:-5].isdigit():
            pulls.append((int(rf[10:-5]), rf))
        elif rf.startswith("refs/heads/"):
            heads.append((rf[11:], rf))
    return (heads, pulls), ""

def _ref_plan(heads, pulls, branch: str):
    plan = []
    if branch:
        rspec = f"refs/pull/{branch}/head" if branch.isdigit() else (branch if branch.startswith("refs/") else f"refs/heads/{branch}")
        plan.append((branch, rspec))
    plan += [(b, r) for b, r in heads if b not in ("main", "master")]
    plan += [(f"PR #{n}", r) for n, r in sorted(pulls, reverse=True)]
    plan += [(b, r) for b, r in heads if b in ("main", "master")]
    return plan or [("HEAD", "HEAD")]

def _repo_help(slug: str, url: str) -> str:
    lines = [f"ไม่พบแพ็กเกจ strata/ ใน '{slug}' ({url})"]
    if _is_kaggle():
        lines += [
            "Kaggle: เปิดอินเทอร์เน็ตก่อน — Settings (แถบขวา) → Internet → On แล้วรันเซลล์นี้อีกครั้ง",
            "Kaggle (ไม่ใช้เน็ต): Add Input → Datasets → แนบ repo นี้ แล้วรันใหม่ ระบบจะอ่านจาก /kaggle/input เอง",
        ]
    elif _is_colab():
        lines += ["Colab: ตรวจว่า runtime ต่อ GitHub ได้ แล้วรันเซลล์นี้ใหม่"]
    lines += [
        f"ตรวจชื่อ repo ให้ถูกแบบ owner/repo (ค่าเริ่มต้นคือ {DEFAULT_SLUG})",
        "repo ต้องเป็น public และมีทั้ง strata/__init__.py + pyproject.toml",
        "ถ้าโค้ดอยู่ในโฟลเดอร์ย่อย ต้องลึกไม่เกิน 2 ชั้น เช่น multiple-projects/strata/",
    ]
    return lines[0] + "".join(f"\n  • {ln}" for ln in lines[1:])

def _target_dir() -> Path:
    if _is_kaggle(): return Path("/kaggle/working/Colab")
    return Path("/content/Colab") if Path("/content").is_dir() else Path.cwd() / "Colab"

def _announce(root: Path, target: Path, source: str) -> Path:
    rel = root.relative_to(target)
    where = str(root) if rel == Path(".") else f"{target.name}/{rel}"
    print(f"✅ ดึงแพ็กเกจ strata/ สำเร็จจาก {source} → {where}", flush=True)
    if not (root / "pyproject.toml").is_file():
        print("⚠️ ไม่มี pyproject.toml ใน project root — bootstrap จะสร้างโปรเจกต์ใหม่ให้อัตโนมัติ", flush=True)
    return root

def _ensure_repo(slug: str, branch: str) -> Path:
    target = _target_dir()
    cached = _find_project_root(target) if not AUTO_UPDATE else None
    if cached is not None:
        print(f"ℹ️ AUTO_UPDATE=False — ใช้โค้ดเดิมที่ {cached}", flush=True)
        return cached
    url = f"https://github.com/{slug}.git"
    if shutil.which("git"):
        refs, err = _list_refs(url)
        if refs is None:
            print(f"⚠️ เข้า repo ไม่ได้: {err.splitlines()[-1] if err else 'unknown error'}", flush=True)
        else:
            for lbl, rspec in _ref_plan(refs[0], refs[1], branch):
                if not _git_fetch_ref(target, url, rspec):
                    continue
                root = _find_project_root(target)
                if root is not None:
                    return _announce(root, target, f"'{lbl}' ({slug})")
                print(f"⚠️ '{lbl}' โคลนได้แต่ไม่พบ strata/ — หยุดลอง ref อื่น (โครงสร้าง repo เหมือนกันทุก ref)", flush=True)
                break
    else:
        print("⚠️ เครื่องนี้ไม่มีคำสั่ง git — เปลี่ยนไปดาวน์โหลด tarball แทน", flush=True)
    if _download_tarball(slug, branch, target):
        root = _find_project_root(target)
        if root is not None:
            return _announce(root, target, f"tarball ({slug})")
        print("⚠️ tarball ที่โหลดมาไม่มี strata/ + pyproject.toml", flush=True)
    raise RuntimeError(_repo_help(slug, url))

print(f"▶️ เริ่มเซลล์ 1 • rev {NOTEBOOK_REV} (python {sys.version.split()[0]})", flush=True)
print(f"🧭 แพลตฟอร์ม: {'🐨 Kaggle' if _is_kaggle() else ('🟡 Colab' if _is_colab() else '💻 Local')}", flush=True)
REPO_ROOT = _sync_from_kaggle() or _ensure_repo(REPO_SLUG.strip() or DEFAULT_SLUG, REPO_BRANCH.strip())
REPO_ROOT = Path(REPO_ROOT)
print(f"📂 project root: {REPO_ROOT}", flush=True)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) in sys.path: sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))
_purge_modules()
try:
    from strata.bootstrap import ensure_environment, prepare_setup_log
except ImportError:
    from strata.bootstrap import ensure_environment, setup_log_path as prepare_setup_log  # type: ignore[assignment]
print(f"📄 setup log: {prepare_setup_log()}", flush=True)
t0 = time.perf_counter()
ensure_environment(nix=USE_NIX)
if _is_colab():
    if Path(".venv/bin/python").is_file() and shutil.which("uv"):
        subprocess.run(["uv", "pip", "uninstall", "--quiet", "--python", ".venv/bin/python", "ipywidgets", "jupyterlab-widgets", "widgetsnbextension", "comm", "ipykernel"], capture_output=True, check=False)
    for _sp in glob.glob("/content/Colab/.venv/lib/python*/site-packages/strata"):
        if Path(_sp).is_dir() and not Path(_sp).is_symlink(): shutil.rmtree(_sp, ignore_errors=True)
    for _vp in [p for p in sys.path if "/.venv/" in p]: sys.path.remove(_vp); sys.path.append(_vp)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf", "fastembed"], check=False)
_purge_modules()
import strata.config as _cfg
from strata.notebook import diagnose, register_magics
from strata.server import install_llama_server
_cfg.COMFY_MODEL_SUBDIRS = ("diffusion_models", "text_encoders", "vae", "loras")
_cfg.SPEC_STATE["mode"] = "auto"; _cfg.SPEC_STATE["kv_q8"] = True
register_magics()
print("📦 โหลดแคตตาล็อกสำเร็จ: 66 โมเดล (54 LLM + 6 Image GGUF + 6 Uncensored LoRA) + 17 Draft", flush=True)
diagnose(); install_llama_server()
print(f"✅ เตรียมสภาพแวดล้อมเสร็จใน {time.perf_counter() - t0:.1f}s — รันเซลล์ที่ 2 ต่อได้เลย!", flush=True)


In [2]:
# @title 2) 🎛️ เปิดหน้าจอ Strata Studio 3 แท็บ (HTML5 ถาวร • 66 โมเดล • ปุ่มรีพาย Pro ลากคลุมข้อความ/ย้อนต้นทาง/ถามต่อ/วาดภาพ + Auto โหนด 4–7) { display-mode: "form" }
UI_MODE = "HTML5"

import base64, concurrent.futures, contextlib, fnmatch, glob, json, os, random, re, shutil, socket, subprocess, sys, threading, time, urllib.request
from pathlib import Path
from IPython import get_ipython
from IPython.display import HTML, JSON, display

socket.setdefaulttimeout(None)
os.environ["HF_HUB_DISABLE_XET"] = "1"
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
SYNC_STATE_FILE = Path("/tmp/_strata_comfy_sync.json")
_has_gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
IS_KAGGLE = any(k.startswith("KAGGLE_") for k in os.environ) or Path("/kaggle/working").is_dir()
IS_COLAB = ("COLAB_RELEASE_TAG" in os.environ) or ("COLAB_GPU" in os.environ and not IS_KAGGLE)
if IS_COLAB:
    with contextlib.suppress(Exception):
        from google.colab import output as _co_fix  # type: ignore[import-not-found]
        if hasattr(_co_fix, "disable_custom_widget_manager"): _co_fix.disable_custom_widget_manager()
    for _sp in glob.glob("/content/Colab/.venv/lib/python*/site-packages/strata"):
        if Path(_sp).is_dir() and not Path(_sp).is_symlink(): shutil.rmtree(_sp, ignore_errors=True)
    for _vp in [p for p in sys.path if "/.venv/" in p]: sys.path.remove(_vp); sys.path.append(_vp)

_bases = [Path("/kaggle/working/Colab"), Path("/content/Colab")]
_cands = [Path.cwd(), *Path.cwd().parents, *_bases]
for _b in _bases: _cands += [*sorted(_b.glob("*/")), *sorted(_b.glob("*/*/"))]  # รองรับ repo ที่ห่อโค้ดไว้ในโฟลเดอร์ย่อย เช่น multiple-projects/
for _base in _cands:
    if (_base / "strata" / "__init__.py").is_file():
        while str(_base) in sys.path: sys.path.remove(str(_base))
        sys.path.insert(0, str(_base)); os.chdir(_base); break
else:
    raise RuntimeError("ยังไม่พบแพ็กเกจ strata/ — กรุณากดรันเซลล์ที่ 1 ก่อนครับ")

import strata.config as _cfg
_cfg.COMFY_MODEL_SUBDIRS = ("diffusion_models", "text_encoders", "vae", "loras")
if not _cfg.SPEC_STATE.get("_user_set_mode"):
    _cfg.SPEC_STATE["mode"] = "auto"; _cfg.SPEC_STATE["kv_q8"] = True

for _rmk in ("flux-3-klein-uncensored", "lora-qwen-lightning-4step", "lora-qwen-next-scene", "lora-qwen-inscene", "lora-flux-realism", "lora-flux-midjourney-mix2"):
    _cfg.MODEL_CATALOG.pop(_rmk, None)
    while _rmk in _cfg.ALL_PRESET_KEYS: _cfg.ALL_PRESET_KEYS.remove(_rmk)

_TEXT_EXTRAS = (
    ("lexi-llama31-8b-v2", "bartowski/Llama-3.1-8B-Lexi-Uncensored-V2-GGUF", ["*Q4_K_M.gguf"], None, 4.9, ["heretic", "rp"], "🔥 Llama-3.1-8B Lexi Uncensored V2 (Q4_K_M)"),
    ("stheno-l31-8b-v34", "bartowski/Llama-3.1-8B-Stheno-v3.4-GGUF", ["*Q4_K_M.gguf"], None, 4.9, ["rp", "heretic"], "🎭 Llama-3.1-8B Stheno v3.4 (Q4_K_M)"),
    ("dolphin3-llama31-8b", "bartowski/Dolphin3.0-Llama3.1-8B-GGUF", ["*Q4_K_M.gguf"], None, 4.9, ["heretic", "rp", "agent"], "🐬 Dolphin 3.0 Llama-3.1-8B Uncensored (Q4_K_M)"),
    ("lumimaid-magnum-v4-12b", "bartowski/Lumimaid-Magnum-v4-12B-GGUF", ["*Q4_K_M.gguf"], None, 7.5, ["rp", "heretic"], "🎭 Lumimaid × Magnum v4 12B (Q4_K_M)"),
    ("wizardlm-13b-uncensored", "mradermacher/WizardLM-13B-Uncensored-i1-GGUF", ["*Q4_K_M.gguf"], None, 7.9, ["heretic", "rp"], "🧙 WizardLM-13B Uncensored imatrix (Q4_K_M)"),
    ("qwen3-14b-abliterated", "bartowski/mlabonne_Qwen3-14B-abliterated-GGUF", ["*Q4_K_M.gguf"], None, 9.0, ["heretic", "rp", "code"], "🔥 Qwen3-14B Abliterated (Q4_K_M)"),
    ("noromaid-20b-uncensored", "mradermacher/Noromaid-20b-v0.1.1-GGUF", ["*Q4_K_M.gguf"], None, 12.1, ["rp", "heretic"], "🎭 Noromaid-20B v0.1.1 (Q4_K_M)"),
    ("cydonia-24b-v43-rp", "bartowski/TheDrummer_Cydonia-24B-v4.3-GGUF", ["*Q4_K_M.gguf"], None, 14.3, ["rp", "heretic"], "🎭 Cydonia-24B v4.3 โดย TheDrummer (Q4_K_M)"),
    ("dolphin-24b-venice", "mradermacher/Dolphin-Mistral-24B-Venice-Edition-GGUF", ["*Q4_K_M.gguf"], None, 14.4, ["heretic", "rp"], "🐬 Dolphin-Mistral-24B Venice Edition (Q4_K_M)"),
    ("mistral-small-24b-abliterated", "bartowski/huihui-ai_Mistral-Small-24B-Instruct-2501-abliterated-GGUF", ["*Q5_K_M.gguf"], None, 16.8, ["heretic", "rp"], "🔥 Mistral-Small-24B Abliterated (Q5_K_M)"),
    ("wizardlm-30b-uncensored", "mradermacher/WizardLM-30B-Uncensored-Guanaco-SuperCOT-30b-GGUF", ["*Q4_K_M.gguf"], None, 19.7, ["heretic", "rp"], "🧙 WizardLM-30B Uncensored SuperCOT (Q4_K_M)"),
    ("qwq-32b-abliterated", "bartowski/huihui-ai_QwQ-32B-abliterated-GGUF", ["*Q4_K_M.gguf"], None, 19.9, ["heretic", "rp", "code"], "🧠 QwQ-32B Reasoning Abliterated (Q4_K_M)"),
    ("noromaid-mixtral-8x7b", "mradermacher/Noromaid-v0.1-mixtral-8x7b-Instruct-v3-i1-GGUF", ["*IQ3_M.gguf"], None, 21.5, ["rp", "heretic"], "🎭 Noromaid Mixtral-8x7B MoE (IQ3_M)"),
    ("euryale-70b-v23-rp", "bartowski/L3.3-70B-Euryale-v2.3-GGUF", ["*IQ2_M.gguf"], None, 24.1, ["rp", "heretic"], "👑 Llama-3.3-70B Euryale v2.3 (IQ2_M)"),
    ("qwen38-27b-obliterated", "mradermacher/Qwen3.8-27B-OBLITERATED-GGUF", ["*Q5_K_M.gguf", "*mmproj-Q8_0.gguf"], ["*f16*"], 20.3, ["obliterated", "heretic", "rp", "code"], "💥 Qwen3.8-27B-OBLITERATED V3 (Q5_K_M + mmproj)"),
    ("gemma4-e4b-obliterated", "rdhorner/gemma-4-E4B-it-OBLITERATED-GGUF", ["*Q4_K_M.gguf", "*mmproj*F16.gguf"], None, 5.8, ["obliterated", "heretic", "rp"], "💥 Gemma-4-E4B-it OBLITERATED v3 (Q4_K_M + Vision)"),
    ("gemma4-e4b-ultra-heretic", "llmfan46/gemma-4-E4B-it-ultra-uncensored-heretic-GGUF", ["*heretic-Q6_K.gguf", "*mmproj-BF16.gguf"], ["*Q4_K*", "*Q5_K*", "*Q8_0*", "*heretic-BF16*"], 7.1, ["heretic", "rp", "agent", "code"], "🔥 Gemma-4-E4B-it Ultra Uncensored Heretic ARA (Q6_K + Vision)"),
    ("glm4-32b-heretic-v2", "mradermacher/GLM-4-32B-0414-uncensored-heretic-v2-GGUF", ["*Q4_K_M.gguf"], None, 19.8, ["heretic", "rp", "code", "agent"], "🔥 GLM-4-32B-0414 Uncensored Heretic v2 ARA (Q4_K_M)"),
    ("gpt-oss-20b-heretic", "mradermacher/gpt-oss-20b-uncensored-heretic-GGUF", ["*Q4_K_M.gguf"], None, 15.8, ["heretic", "rp", "code", "agent"], "🔥 OpenAI gpt-oss-20b Uncensored Heretic (Q4_K_M • MoE)"),
    ("stheno-l2-13b", "TheBloke/Stheno-L2-13B-GGUF", ["*Q4_K_M.gguf", "*q4_K_M.gguf"], None, 7.9, ["rp", "heretic"], "🎭 Stheno-L2-13B Sao10K × TheBloke (Q4_K_M)"),
    ("qwen38-9b-heretic", "optionalAI/Qwen3.8-9B-heretic-uncensored-Q4_K_M-GGUF", ["*q4_k_m.gguf", "*Q4_K_M.gguf"], None, 5.6, ["heretic", "rp", "code", "agent"], "🔥 Qwen3.8-9B Heretic Uncensored (Q4_K_M)"),
    ("wizardlm-13b-thebloke", "TheBloke/WizardLM-13B-Uncensored-GGUF", ["*Q4_K_M.gguf"], None, 7.9, ["heretic", "rp"], "🧙 WizardLM-13B-Uncensored TheBloke (Q4_K_M)"),
    ("qwen3-8b-hivemind-unrestricted", "DavidAU/Qwen3-8B-Hivemind-Instruct-Heretic-Abliterated-Uncensored-NEO-Imatrix-GGUF", ["*Q6_K*.gguf"], ["*mmproj*"], 6.8, ["unrestricted", "heretic", "rp", "code"], "🔓 Qwen3-8B-Hivemind Unrestricted NEO (Q6_K • 256k)"),
    ("gemma3-4b-unrestricted", "mradermacher/gemma-3-4b-it-uncensored-GGUF", ["*Q8_0.gguf", "*mmproj-Q8_0.gguf"], ["*f16*"], 4.9, ["unrestricted", "heretic", "rp"], "🔓 Gemma-3-4B-IT Unrestricted Multimodal (Q8_0)"),
    ("skyfall-31b-v42-rp", "mradermacher/Skyfall-31B-v4.2-GGUF", ["*Q4_K_M.gguf"], None, 19.1, ["rp"], "🎭 Skyfall-31B v4.2 โดย TheDrummer (Q4_K_M)"),
    ("magnum-v4-22b-rp", "mradermacher/magnum-v4-22b-i1-GGUF", ["*i1-Q6_K.gguf"], None, 18.4, ["rp"], "🎭 Magnum-v4-22B โดย Anthracite (Q6_K)"),
    ("rocinante-12b-v11-rp", "bartowski/Rocinante-12B-v1.1-GGUF", ["Rocinante-12B-v1.1-Q6_K.gguf"], ["*Q6_K_L*"], 10.1, ["rp"], "🎭 Rocinante-12B v1.1 โดย TheDrummer (Q6_K)"),
    ("nemomix-unleashed-12b-rp", "bartowski/NemoMix-Unleashed-12B-GGUF", ["NemoMix-Unleashed-12B-Q6_K.gguf"], ["*Q6_K_L*"], 10.1, ["rp", "unrestricted"], "🎭🔓 NemoMix-Unleashed-12B (Q6_K)"),
    ("anubis-70b-v12-rp", "bartowski/TheDrummer_Anubis-70B-v1.2-GGUF", ["*IQ2_M.gguf"], None, 24.1, ["rp"], "👑 Anubis-70B v1.2 โดย TheDrummer (IQ2_M)"),
    ("nevoria-70b-rp", "bartowski/L3.3-MS-Nevoria-70b-GGUF", ["*IQ2_M.gguf"], None, 24.1, ["rp"], "👑 L3.3-MS-Nevoria-70B โดย Steelskull (IQ2_M)"),
    ("lumimaid-v02-8b-rp", "mradermacher/Lumimaid-v0.2-8B-GGUF", ["*Q6_K.gguf"], None, 6.6, ["rp"], "🎭 Lumimaid-v0.2-8B NeverSleep (Q6_K)"),
    ("lumimaid-v02-70b-rp", "mradermacher/Lumimaid-v0.2-70B-i1-GGUF", ["*IQ2_M.gguf"], None, 24.1, ["rp"], "👑 Lumimaid-v0.2-70B NeverSleep (IQ2_M)"),
    ("dolphin3-r1-24b", "eaddario/Dolphin3.0-R1-Mistral-24B-GGUF", ["*Q5_K_M.gguf"], None, 16.9, ["heretic", "rp", "code", "agent"], "🐬 Dolphin 3.0 R1 Mistral-24B (Q5_K_M)"),
    ("gemma4-31b-heretic", "Stabhappy/gemma-4-31B-it-heretic-Gguf", ["*Q4_K_M*.gguf"], ["*F32*"], 18.7, ["heretic", "rp", "agent"], "🔥 Gemma-4-31B-it Heretic Dense (Q4_K_M)")
)
for _k, _r, _inc, _exc, _gb, _tg, _ds in _TEXT_EXTRAS:
    _cfg.MODEL_CATALOG[_k] = {"repo_id": _r, "include": _inc, "exclude": _exc, "approx_gb": _gb, "has_mtp": False, "tags": _tg, "desc": _ds}
    if _k not in _cfg.ALL_PRESET_KEYS: _cfg.ALL_PRESET_KEYS.append(_k)

_IMAGE_EXTRAS = {
    "z-image-turbo-heretic": {"repo_id": "unsloth/Z-Image-Turbo-GGUF", "kind": "image", "include": ["z-image-turbo-Q4_K_M.gguf", "DreamFast-qwen3-4b-heretic.i1-Q4_K_M.gguf", "split_files/vae/ae.safetensors"], "targets": {"z-image-turbo-Q4_K_M.gguf": "diffusion_models", "DreamFast-qwen3-4b-heretic.i1-Q4_K_M.gguf": "text_encoders", "split_files/vae/ae.safetensors": "vae"}, "sources": {"DreamFast-qwen3-4b-heretic.i1-Q4_K_M.gguf": "mradermacher/DreamFast-qwen3-4b-heretic-i1-GGUF", "split_files/vae/ae.safetensors": "Comfy-Org/z_image_turbo"}, "exclude": None, "approx_gb": 6.6, "has_mtp": False, "tags": ["heretic"], "desc": "🔥 Z-Image-Turbo + Qwen3-4B Heretic GGUF (9 steps • เร็วสุดบน T4)"},
    "qwen-image-21-heretic": {"repo_id": "abenzerps/Qwen-Image-2.1-Turbo-Quantized", "kind": "image", "include": ["qwen-image-2.1-turbo-Q4_K_M.gguf", "comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors", "vae/qwen_image_2.1_vae_bf16.safetensors"], "targets": {"qwen-image-2.1-turbo-Q4_K_M.gguf": "diffusion_models", "comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "sources": {"comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors": "DreamFast/Qwen3-VL-8B-Heretic-1.3.0"}, "exclude": None, "approx_gb": 14.7, "has_mtp": False, "tags": ["heretic"], "desc": "🔥 Qwen-Image-2.1 Turbo + Heretic v1.3 Encoder (8 steps)"},
    "qwen-image-21-heretic-q6": {"repo_id": "abenzerps/Qwen-Image-2.1-Uncensored-GGUF", "kind": "image", "include": ["qwen-image-2.1-UC-Q6_K.gguf", "comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors", "vae/qwen_image_2.1_vae_bf16.safetensors"], "targets": {"qwen-image-2.1-UC-Q6_K.gguf": "diffusion_models", "comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "sources": {"comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors": "DreamFast/Qwen3-VL-8B-Heretic-1.3.0", "vae/qwen_image_2.1_vae_bf16.safetensors": "abenzerps/Qwen-Image-2.1-Turbo-Quantized"}, "exclude": None, "approx_gb": 26.8, "has_mtp": False, "tags": ["heretic"], "desc": "🔥 Qwen-Image-2.1 UC Q6_K + Heretic v1.3 (คมชัดสูงสุด 2×T4)"},
    "chroma1-hd-uncensored": {"repo_id": "silveroxides/Chroma1-HD-GGUF", "kind": "image", "include": ["Chroma1-HD-Q4_K_M.gguf", "t5xxl_fp8_e4m3fn_scaled.safetensors", "split_files/vae/ae.safetensors"], "targets": {"Chroma1-HD-Q4_K_M.gguf": "diffusion_models", "t5xxl_fp8_e4m3fn_scaled.safetensors": "text_encoders", "split_files/vae/ae.safetensors": "vae"}, "sources": {"t5xxl_fp8_e4m3fn_scaled.safetensors": "comfyanonymous/flux_text_encoders", "split_files/vae/ae.safetensors": "Comfy-Org/z_image_turbo"}, "exclude": None, "approx_gb": 11.1, "has_mtp": False, "tags": ["heretic"], "desc": "🎨 Chroma1-HD Uncensored GGUF (8.9B FLUX • รับ Negative Prompt เต็มรูปแบบ)"},
    "qwen-image-21-turbo": {"repo_id": "abenzerps/Qwen-Image-2.1-Turbo-Quantized", "kind": "image", "include": ["qwen-image-2.1-turbo-Q4_K_M.gguf", "text_encoders/qwen3vl_8b_int8_convrot.safetensors", "vae/qwen_image_2.1_vae_bf16.safetensors"], "targets": {"qwen-image-2.1-turbo-Q4_K_M.gguf": "diffusion_models", "text_encoders/qwen3vl_8b_int8_convrot.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "exclude": None, "approx_gb": 14.6, "has_mtp": False, "tags": ["heretic"], "desc": "⚡ Qwen-Image-2.1 Turbo Uncensored (8 steps • Q4_K_M + Qwen3VL-8B)"},
    "qwen-image-21-uncensored": {"repo_id": "abenzerps/Qwen-Image-2.1-Uncensored-GGUF", "kind": "image", "include": ["qwen-image-2.1-UC-Q4_K_M.gguf", "text_encoders/qwen3vl_8b_int8_convrot.safetensors", "vae/qwen_image_2.1_vae_bf16.safetensors"], "targets": {"qwen-image-2.1-UC-Q4_K_M.gguf": "diffusion_models", "text_encoders/qwen3vl_8b_int8_convrot.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "sources": {"text_encoders/qwen3vl_8b_int8_convrot.safetensors": "abenzerps/Qwen-Image-2.1-Turbo-Quantized", "vae/qwen_image_2.1_vae_bf16.safetensors": "abenzerps/Qwen-Image-2.1-Turbo-Quantized"}, "exclude": None, "approx_gb": 22.1, "has_mtp": False, "tags": ["heretic"], "desc": "🎨 Qwen-Image-2.1 Uncensored Full CFG (Q4_K_M + Qwen3VL-8B + VAE)"}
}
for _ik, _iv in _IMAGE_EXTRAS.items():
    _cfg.MODEL_CATALOG[_ik] = _iv
    if _ik not in _cfg.ALL_PRESET_KEYS: _cfg.ALL_PRESET_KEYS.append(_ik)

_UNCEN_LORAS = (
    ("lora-qwen21-uncensored", "JoyFusionAI/Qwen-Image-2.1-Uncensored-LoRA", ["*.safetensors"], "qwen-image-2.1-uncensored-lora.safetensors", 1.0, 0.15, "🔥🧩 [Uncensored LoRA] Qwen-Image-2.1 Uncensored (Rank-16 BF16 คู่ Qwen-Image 2.1)"),
    ("lora-qwen21-omni-uncensored", "chatrlabs/Qwen-Image-2.1-Uncensored-Omni-LoRA-v0.1", ["*.safetensors"], "chatrlabs-omni-lora-v0.1-bf16.safetensors", 1.0, 0.67, "🔥🧩 [Uncensored LoRA] ChatR Qwen-Image-2.1 Uncensored Omni v0.1 (Rank-128 671MB)"),
    ("lora-zimage-turbo-uncensored", "torestinbar/z-image-turbo", ["loras/zimage_cos-NSFW-lora.safetensors", "*lora*.safetensors"], "zimage-turbo-uncensored-lora.safetensors", 0.85, 0.16, "🔥🧩 [Uncensored LoRA] Z-Image-Turbo Uncensored (ปลดล็อกเต็มรูปแบบคู่ Z-Image-Turbo)"),
    ("lora-flux-uncensored-v2", "Shar514/Flux-Uncensored-V2", ["*.safetensors"], "flux-uncensored-v2.safetensors", 0.85, 0.38, "🔥🧩 [Uncensored LoRA] FLUX Uncensored V2 (BF16 เสถียรสูงคู่ Chroma1-HD / FLUX)"),
    ("lora-flux-uncensored-sagalink", "Sagalink/Flux-uncensored", ["*.safetensors"], "sagalink-flux-uncensored.safetensors", 0.85, 0.69, "🔥🧩 [Uncensored LoRA] Sagalink FLUX Uncensored Full (BF16 687MB คู่ Chroma1-HD / FLUX)"),
    ("lora-flux-uncensored-lite", "salomonsky/flux-lora-uncensored", ["*.safetensors"], "flux-lora-uncensored-lite.safetensors", 0.85, 0.04, "🔥🧩 [Uncensored LoRA] FLUX Uncensored Lite (~40MB โหลดไวคู่ Chroma1-HD / FLUX)")
)
for _lk, _lr, _linc, _lf, _ls, _lgb, _lds in _UNCEN_LORAS:
    _cfg.MODEL_CATALOG[_lk] = {"repo_id": _lr, "kind": "lora", "include": _linc, "lora_file": _lf, "default_strength": _ls, "exclude": None, "approx_gb": _lgb, "has_mtp": False, "tags": ["lora", "heretic", "unrestricted"], "desc": _lds}
    if _lk not in _cfg.ALL_PRESET_KEYS: _cfg.ALL_PRESET_KEYS.append(_lk)

_cfg.IMAGE_PRESET_KEYS = [k for k in _cfg.ALL_PRESET_KEYS if (_cfg.MODEL_CATALOG.get(k) or {}).get("kind") == "image"]
_cfg.LORA_PRESET_KEYS = [k for k in _cfg.ALL_PRESET_KEYS if (_cfg.MODEL_CATALOG.get(k) or {}).get("kind") == "lora"]
_cfg.TEXT_PRESET_KEYS = [k for k in _cfg.ALL_PRESET_KEYS if (_cfg.MODEL_CATALOG.get(k) or {}).get("kind") not in ("image", "lora")]

_DRAFTS_EXT = (
    ("gemma4-26b-eagle3", "williamliao/gemma-4-26B-A4B-it-speculator.eagle3-F16-GGUF", ["*Q4_K_M*.gguf", "*Q8_0*.gguf"], False, True, "gemma", "🦅 Gemma-4 26B EAGLE-3"),
    ("qwen3-14b-eagle3", "williamliao/Qwen3-14B-EAGLE3-Speculator-GGUF", ["*eagle3-Q4_K_M.gguf"], False, True, "qwen", "🦅 Qwen3-14B EAGLE-3"),
    ("qwen38-coletti-mtp", "JonathanColetti/Qwen3.8-27B-Uncensored-GGUF", ["*draft*Q8_0*", "*draft*Q4*"], True, False, "qwen", "🚀 Qwen3.8-27B MTP Draft"),
    ("qwen35-0.8b-heretic", "mradermacher/Qwen3.5-0.8B-heretic-v3-i1-GGUF", ["*Q4_K_M.gguf"], False, False, "qwen", "🔥 Qwen3.5-0.8B Heretic (~0.5GB)"),
    ("qwen3-0.6b-heretic", "mradermacher/Qwen3.0-6B-heretic-abliterated-uncensored-i1-GGUF", ["*Q4_K_M.gguf"], False, False, "qwen", "🔥 Qwen3.0-6B Heretic (~0.4GB)"),
    ("qwen3-coder-0.8b", "DavidAU/Qwen3-Zero-Coder-Reasoning-V2-0.8B-NEO-EX-GGUF", ["*Q4_K_M*.gguf", "*IQ4_XS*.gguf"], False, False, "qwen", "💻 Qwen3-Zero-Coder-0.8B (~0.5GB)"),
    ("llama32-1b-uncensored", "mradermacher/Llama-3.2-1B-Instruct-Uncensored-GGUF", ["*Q4_K_M.gguf"], False, False, "llama3", "🔥 Llama-3.2-1B Uncensored (~0.8GB)"),
    ("ministral-3b-abliterated", "Abiray/Huihui-Ministral-3B-Instruct-2512-abliterated-GGUF", ["*Q4_K_M*.gguf"], False, False, "tekken", "🔥 Ministral-3B Abliterated (~1.9GB)"),
    ("tinyllama-1.1b-heretic", "mradermacher/TinyLlama-1.1B-Chat-v1.0-Heretic-GGUF", ["*Q4_K_M.gguf"], False, False, "llama2", "🔥 TinyLlama-1.1B Heretic (~0.7GB)"),
    ("gemma3-1b-heretic", "mradermacher/gemma-3-1b-it-heretic-abliterated-uncensored-i1-GGUF", ["*Q4_K_M.gguf"], False, False, "gemma", "🔥 Gemma-3-1B Heretic (~0.8GB)")
)
for _dk, _dr, _dinc, _dm, _de, _dfam, _dds in _DRAFTS_EXT:
    _cfg.DRAFT_CATALOG[_dk] = {"repo_id": _dr, "include": _dinc, "exclude": ["*mmproj*", "*F16*"], "is_mtp": _dm, "is_eagle3": _de, "vocab_family": _dfam, "desc": _dds}

import strata.download as _dl, strata.imagegen as _ig, strata.server as _srv, strata.utils as _ut
from rich.text import Text as _RichText
from strata.platforms import detect as _detect_plat
from strata.rag import GLOBAL_RAG

_ig.COMFY_MODEL_SUBDIRS = ("diffusion_models", "text_encoders", "vae", "loras")
_check_server_online = _srv._check_server_online
_invalidate_server_cache = getattr(_srv, "_invalidate_server_cache", lambda *_a, **_k: None)
check_gpu_and_recommend = _srv.check_gpu_and_recommend
get_openai_client = _srv.get_openai_client
install_llama_server = _srv.install_llama_server
start_strata_server = _srv.start_strata_server
stop_strata_server = _srv.stop_strata_server
_dir_stats, _fmt_bytes, _fmt_rate = _ut._dir_stats, _ut._fmt_bytes, _ut._fmt_rate
_invalidate_dir_cache = getattr(_ut, "_invalidate_dir_cache", lambda: None)
_remember = getattr(_ut, "_remember", lambda _m: None)
net_bandwidth = getattr(_ut, "net_bandwidth", lambda: {"down_bps": 0.0, "up_bps": 0.0})

_ST = {"job": "", "live_step": "", "notice": "", "kind": "info", "history": [], "img_stat": "", "last_img": None}

_TH_VISUAL_MAP = (
    ("หญิงสาว", "young woman"), ("ผู้หญิง", "woman"), ("สาวสวย", "beautiful woman"), ("ผู้ชาย", "man"), ("ชายหนุ่ม", "young man"),
    ("นักรบ", "warrior"), ("เจ้าหญิง", "princess"), ("แม่มด", "sorceress"), ("ซามูไร", "samurai"), ("นินจา", "ninja"),
    ("ไซเบอร์พังก์", "cyberpunk"), ("อนิเมะ", "anime style"), ("มังงะ", "manga illustration"), ("สมจริง", "photorealistic, raw photo"),
    ("ภาพถ่าย", "DSLR photograph, 85mm lens"), ("ภาพยนตร์", "cinematic film still, anamorphic"), ("แฟนตาซี", "dark fantasy art"),
    ("ดาร์ก", "dark moody atmosphere"), ("แสงนีออน", "glowing neon lights"), ("สายฝน", "rain, wet reflections"),
    ("กลางคืน", "night scene, moonlight"), ("พระอาทิตย์ตก", "golden hour sunset lighting"), ("หมอก", "volumetric fog"),
    ("ทะเล", "ocean waves, coastal"), ("ภูเขา", "majestic mountains"), ("ป่า", "lush forest"), ("เมือง", "detailed cityscape"),
    ("อวกาศ", "cosmic sci-fi space"), ("ชุดเกราะ", "intricate armor"), ("กิโมโน", "traditional kimono"), ("ชุดไทย", "traditional Thai attire"),
    ("เต็มตัว", "full body shot"), ("ครึ่งตัว", "medium portrait shot"), ("โคลสอัป", "detailed close-up portrait"), ("มุมกว้าง", "wide angle shot"),
)
_STYLE_PRESETS = {
    "portrait_photo": ("📸 Portrait Photoreal", "photorealistic, natural skin texture, subsurface scattering, 85mm f/1.4 lens, cinematic rim lighting, sharp focus, 8k", "blurry, low quality, plastic skin, waxy, CGI, 3d render, bad anatomy, deformed hands, extra fingers, missing fingers, cross-eyed, watermark, text, signature"),
    "anime_illust": ("🎨 Anime / Illustration", "masterpiece anime illustration, crisp lineart, vibrant cel shading, expressive eyes, dynamic lighting, high detail", "photorealistic, 3d render, live action, blurry, lowres, bad anatomy, bad hands, extra digits, fewer digits, jpeg artifacts, watermark, text"),
    "cinematic_scifi": ("🎬 Cinematic / Cyberpunk", "cinematic movie still, volumetric lighting, raytracing reflections, dramatic contrast, anamorphic bokeh, ultra detailed, 8k", "flat lighting, overexposed, amateur, blurry, low quality, deformed, bad anatomy, watermark, text, logo, cropped"),
    "dark_fantasy": ("🐉 Dark Fantasy", "epic dark fantasy masterpiece, intricate details, dramatic chiaroscuro lighting, atmospheric mist, rich textures, 8k", "modern, bright cartoon, flat colors, blurry, low quality, bad anatomy, deformed limbs, extra fingers, watermark, text"),
    "landscape_nature": ("🏞️ Landscape / Scenery", "breathtaking wide angle landscape photography, golden hour volumetric rays, crisp foreground and atmospheric depth, 8k", "blurry, low resolution, hazy, oversaturated, distorted perspective, watermark, text, frame"),
    "general_art": ("✨ High-Detail Visual", "masterpiece, ultra detailed, dramatic lighting, sharp focus, balanced composition, high resolution", "blurry, low quality, deformed, bad anatomy, disfigured, poorly drawn hands, extra limbs, watermark, text, signature"),
}

def _merge_reply_and_prompt(reply_text: str, user_text: str) -> str:
    rt = re.sub(r"\s+", " ", (reply_text or "").strip())[:420]
    ut = (user_text or "").strip()
    return f"{rt}, {ut}" if (rt and ut) else (ut or rt)

def _auto_build_workflow(raw_prompt: str, model_key: str, lora_name: str = "", lora_str: float = 1.0, use_llm: bool = False, port: int = 18080) -> dict:
    txt = (raw_prompt or "").strip()
    user_neg = [m.group(1).strip(" ,.") for m in re.finditer(r"(?:--no|ไม่เอา)\s+([^\-\n]+)", txt, flags=re.IGNORECASE)]
    txt_clean = re.sub(r"(?:--no|ไม่เอา)\s+[^\-\n]+", "", txt, flags=re.IGNORECASE)
    ov_steps = ov_cfg = ov_seed = None
    if (ms := re.search(r"--steps\s+(\d+)", txt_clean, re.I)): ov_steps = max(1, min(60, int(ms.group(1)))); txt_clean = txt_clean.replace(ms.group(0), "")
    if (mc := re.search(r"--cfg\s+([0-9.]+)", txt_clean, re.I)): ov_cfg = max(1.0, min(15.0, float(mc.group(1)))); txt_clean = txt_clean.replace(mc.group(0), "")
    if (md := re.search(r"--seed\s+(-?\d+)", txt_clean, re.I)): ov_seed = int(md.group(1)); txt_clean = txt_clean.replace(md.group(0), "")
    txt_clean = re.sub(r"\s+", " ", txt_clean).strip(); low = txt_clean.lower()
    if any(k in low for k in ("อนิเมะ", "มังงะ", "การ์ตูน", "anime", "manga", "cel shad", "2d", "waifu")): skey = "anime_illust"
    elif any(k in low for k in ("ไซเบอร์พังก์", "ไซไฟ", "นีออน", "อวกาศ", "หุ่นยนต์", "cyberpunk", "sci-fi", "neon", "futuristic", "cinematic", "ภาพยนตร์")): skey = "cinematic_scifi"
    elif any(k in low for k in ("แฟนตาซี", "มังกร", "เวทมนตร์", "นักรบ", "ปีศาจ", "ปราสาท", "fantasy", "dragon", "warrior", "magic", "gothic")): skey = "dark_fantasy"
    elif any(k in low for k in ("วิว", "ภูเขา", "ทะเล", "ธรรมชาติ", "ป่า", "landscape", "scenery", "mountain", "ocean", "panoram")) and not any(p in low for p in ("หญิง", "ชาย", "สาว", "girl", "woman", "man", "portrait")): skey = "landscape_nature"
    elif any(k in low for k in ("หญิง", "ชาย", "สาว", "คน", "พอร์ตเทรต", "ภาพถ่าย", "สมจริง", "portrait", "woman", "girl", "man", "photo", "realistic", "face")): skey = "portrait_photo"
    else: skey = "general_art"
    style_lbl, pos_boost, base_neg = _STYLE_PRESETS[skey]
    if any(k in low for k in ("เร็ว", "ด่วน", "ร่าง", "768", "draft", "quick")): w, h = 768, 768
    elif any(k in low for k in ("แนวนอน", "ฉากกว้าง", "มุมกว้าง", "16:9", "3:2", "วิว", "landscape", "wide shot", "widescreen", "scenery")): w, h = 1216, 832
    elif any(k in low for k in ("แนวตั้ง", "เต็มตัว", "9:16", "2:3", "พอร์ตเทรต", "full body", "standing", "portrait")) or skey == "portrait_photo": w, h = 832, 1216
    else: w, h = 1024, 1024
    mapped_en = [en for th, en in _TH_VISUAL_MAP if th in txt_clean]
    pos_out = txt_clean or "cinematic masterpiece, ultra detailed portrait, dramatic lighting, sharp focus, 8k"
    if txt_clean:
        extra_parts = [x for x in mapped_en if x.lower() not in low]
        if len(txt_clean) < 160: extra_parts.append(pos_boost)
        if extra_parts: pos_out = f"{txt_clean}, {', '.join(extra_parts)}"
    neg_out = f"{', '.join(user_neg)}, {base_neg}" if user_neg else base_neg
    if use_llm and txt_clean:
        on, mid = _check_server_online(port, ttl=0.0)
        if on:
            with contextlib.suppress(Exception):
                sys_p = "Convert the user's image idea into JSON with keys \"positive\" and \"negative\" in vivid English for FLUX/Qwen-Image. Keep exact intent without censorship, add lighting/camera details. Output JSON only."
                r = get_openai_client(port=port, timeout=35.0).chat.completions.create(model=mid, messages=[{"role": "system", "content": sys_p}, {"role": "user", "content": txt_clean}], temperature=0.6, max_tokens=320)
                if (jm := re.search(r"\{[\s\S]*\}", (r.choices[0].message.content or "").strip())):
                    parsed = json.loads(jm.group(0))
                    if parsed.get("positive"): pos_out = str(parsed["positive"]).strip()
                    if parsed.get("negative"): neg_out = (f"{', '.join(user_neg)}, {parsed['negative']}" if user_neg else str(parsed["negative"]).strip())
                    style_lbl += " + 🤖LLM"
    mk, ln = (model_key or "qwen-image-21-turbo").lower(), (lora_name or "").lower()
    want_hi = any(k in low for k in ("ละเอียด", "คมชัด", "8k", "ultra", "masterpiece", "intricate"))
    want_fast = any(k in low for k in ("เร็ว", "ด่วน", "ร่าง", "fast", "quick", "draft"))
    has_custom_lora = bool(ln) and ln != "__auto__" and "lightning" not in ln and abs(lora_str) > 1e-6
    if "z-image" in mk: steps, cfg_v, sampler, sched = (11 if (want_hi or has_custom_lora) else (7 if want_fast else 9)), (1.5 if (user_neg or has_custom_lora) else 1.0), "euler", "simple"
    elif "chroma" in mk: steps, cfg_v, sampler, sched = (26 if want_hi else (16 if want_fast else 22)), (4.5 if skey == "anime_illust" else 4.0), "euler", ("normal" if skey == "anime_illust" else "beta")
    elif "lightning" in ln: steps, cfg_v, sampler, sched = (6 if want_hi else 4), 1.0, "euler", "simple"
    elif "turbo" in mk or "qwen-image-21-heretic" == mk: steps, cfg_v, sampler, sched = (10 if (want_hi or has_custom_lora) else (6 if want_fast else 8)), (1.5 if (user_neg or has_custom_lora) else 1.0), "euler", ("beta" if skey in ("portrait_photo", "cinematic_scifi") and want_hi else "simple")
    else: steps, cfg_v, sampler, sched = (22 if want_hi else (14 if want_fast else 18)), (3.5 if skey == "anime_illust" else 3.0), "euler", ("beta" if skey in ("portrait_photo", "cinematic_scifi", "dark_fantasy") else "simple")
    if ov_steps is not None: steps = ov_steps
    if ov_cfg is not None: cfg_v = ov_cfg
    seed = ov_seed if ov_seed is not None else -1
    summary = f"{style_lbl} • {w}×{h} • {steps} steps • CFG {cfg_v:.1f} • {sampler}/{sched}"
    return {"raw": txt, "pos": pos_out, "neg": neg_out, "width": w, "height": h, "steps": steps, "cfg": round(cfg_v, 2), "sampler": sampler, "scheduler": sched, "seed": seed, "style_label": style_lbl, "summary": summary, "wf_ts": time.time()}

def _read_sync_state() -> dict:
    if SYNC_STATE_FILE.is_file():
        with contextlib.suppress(Exception): return json.loads(SYNC_STATE_FILE.read_text(encoding="utf-8"))
    return {"active_model": "", "active_lora": "", "lora_strength": 1.0, "workflow": None, "ts": 0.0}

def _write_sync_state(**updates) -> dict:
    cur = _read_sync_state(); cur.update(updates); cur["ts"] = time.time()
    with contextlib.suppress(Exception): SYNC_STATE_FILE.write_text(json.dumps(cur, ensure_ascii=False), encoding="utf-8")
    return cur

def _ui_log_sink(*args, **_kwargs) -> None:
    s = re.sub(r"\x1b\[[0-9;]*m", "", " ".join(str(a) for a in args))
    with contextlib.suppress(Exception): s = _RichText.from_markup(s).plain
    lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
    if lines: _ST["live_step"] = lines[-1]; _remember(lines[-1])

class _SilentConsole:
    def __init__(self, *_a, **_k): pass
    def print(self, *args, **_kwargs): _ui_log_sink(*args)

for _mod in (_srv, _dl, _ig):
    if hasattr(_mod, "rprint"): setattr(_mod, "rprint", _ui_log_sink)
    if hasattr(_mod, "Console"): setattr(_mod, "Console", _SilentConsole)
    setattr(_mod, "print", _ui_log_sink)

def _free_comfy_vram_for_llm() -> None:
    with contextlib.suppress(Exception):
        if len(_srv.gpu_devices(ttl=0.0)) <= 1:
            urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/free", data=b'{"unload_models":true,"free_memory":true}', headers={"Content-Type": "application/json"}), timeout=2.5)

def _free_single_gpu_for_comfy() -> None:
    with contextlib.suppress(Exception):
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/interrupt", data=b"{}", headers={"Content-Type": "application/json"}), timeout=1.5)
    with contextlib.suppress(Exception):
        if len(_srv.gpu_devices(ttl=0.0)) <= 1 and _check_server_online(int(_cfg.DEFAULT_PORT), ttl=0.0)[0]:
            _ST["live_step"] = "🧹 การ์ดเดี่ยว: พัก llama-server คืน VRAM 100% ให้ ComfyUI..."
            stop_strata_server(int(_cfg.DEFAULT_PORT)); _srv.release_gpu_memory(int(_cfg.DEFAULT_PORT)); time.sleep(1.0)

def _infer_model_vocab(key: str) -> str:
    hk = f"{key} {(_cfg.MODEL_CATALOG.get(key) or {}).get('repo_id', '')}".lower()
    if any(x in hk for x in ("lumimaid-v02", "lumimaid-v0.2", "anubis", "nevoria", "lexi-llama31", "stheno-l31", "dolphin3-llama", "euryale", "llama-3.", "llama3.")): return "llama3"
    if any(x in hk for x in ("lumimaid-magnum", "cydonia", "skyfall", "magnum", "rocinante", "nemomix", "dolphin3-r1", "dolphin-24b", "venice", "mistral-small")): return "tekken"
    if any(x in hk for x in ("stheno-l2", "noromaid", "wizardlm", "mixtral")): return "llama2"
    if any(x in hk for x in ("qwen", "qwq", "deepseek-v4-pro-4b", "ornith")): return "qwen"
    if any(x in hk for x in ("gemma", "muse-glimmer", "glimmer")): return "gemma"
    return ""

def _pick_auto_draft_key(key: str) -> str | None:
    cfg = _cfg.MODEL_CATALOG.get(key) or {}
    if cfg.get("kind") in ("image", "lora"): return None
    hk = key.lower()
    if hk == "qwen38-27b-coletti": return "qwen38-coletti-mtp"
    if hk == "qwen3-14b-abliterated": return "qwen3-14b-eagle3"
    if "gemma4-26b" in hk: return "gemma4-26b-eagle3"
    fam = _infer_model_vocab(key)
    if fam == "qwen":
        if "code" in (cfg.get("tags") or []) or "coder" in hk or "cyber" in hk: return "qwen3-coder-0.8b"
        return "qwen35-0.8b-heretic" if any(x in hk for x in ("qwen35", "qwen36", "qwen38", "deepseek-v4-pro-4b")) else "qwen3-0.6b-heretic"
    return {"llama3": "llama32-1b-uncensored", "tekken": "ministral-3b-abliterated", "llama2": "tinyllama-1.1b-heretic", "gemma": "gemma3-1b-heretic"}.get(fam)

def _find_draft_file(sub: Path) -> Path | None:
    if not sub.is_dir(): return None
    hits = sorted(p for p in sub.rglob("*.gguf") if p.is_file() and not p.name.endswith(".part") and "mmproj" not in p.name.lower() and p.stat().st_size > 0)
    return hits[0] if hits else None

def _resolve_draft_for_run(key: str) -> str:
    dk = _pick_auto_draft_key(key)
    if dk and (fp := _find_draft_file(Path(_cfg.DRAFT_DIR) / dk)): return str(fp)
    fam = _infer_model_vocab(key)
    for ck, dv in _cfg.DRAFT_CATALOG.items():
        if fam and dv.get("vocab_family") == fam and not dv.get("is_eagle3") and (fp := _find_draft_file(Path(_cfg.DRAFT_DIR) / ck)): return str(fp)
    return ""

def _resolve_local_target(sub: Path, rel: str) -> Path | None:
    if not sub.is_dir(): return None
    for cand in (sub / rel, sub / Path(rel).name):
        if cand.is_file() and not cand.name.endswith(".part") and cand.stat().st_size > 0: return cand.resolve()
    bn = Path(rel).name.lower()
    return next((p.resolve() for p in sub.rglob("*") if p.is_file() and not p.name.endswith(".part") and p.name.lower() == bn and p.stat().st_size > 0), None)

def _sync_lora_preset(key: str, model_dir: str) -> tuple[bool, str]:
    cfg, sub = _cfg.MODEL_CATALOG.get(key) or {}, Path(model_dir) / key
    if not sub.is_dir(): return False, ""
    sfs = sorted(p for p in sub.rglob("*.safetensors") if p.is_file() and not p.name.endswith(".part") and p.stat().st_size > 0)
    if not sfs: return False, ""
    dst_name = str(cfg.get("lora_file") or sfs[0].name)
    dst = Path(_cfg.COMFY_DIR) / "models" / "loras" / dst_name
    dst.parent.mkdir(parents=True, exist_ok=True)
    with contextlib.suppress(OSError):
        if dst.is_symlink() or dst.exists(): dst.unlink()
        os.symlink(sfs[0].resolve(), dst)
    return True, dst_name

def _sync_image_preset(key: str, model_dir: str) -> tuple[bool, list[str]]:
    cfg, sub = _cfg.MODEL_CATALOG.get(key) or {}, Path(model_dir) / key
    tgts = cfg.get("targets") or {}
    if not sub.is_dir() or not tgts: return False, [Path(r).name for r in tgts]
    missing, cdir = [], Path(_cfg.COMFY_DIR)
    for rel, role in tgts.items():
        found = _resolve_local_target(sub, rel)
        if found is None and role == "diffusion_models":
            gg = [p.resolve() for p in sub.rglob("*.gguf") if p.is_file() and not p.name.endswith(".part") and p.stat().st_size > 100 * 1024**2]
            if gg: found = gg[0]
        if found is None: missing.append(Path(rel).name); continue
        for r_name in [role, *(["clip"] if role == "text_encoders" else (["unet"] if role == "diffusion_models" else []))]:
            dst = cdir / "models" / r_name / Path(rel).name
            dst.parent.mkdir(parents=True, exist_ok=True)
            with contextlib.suppress(OSError):
                if dst.is_symlink() or dst.exists(): dst.unlink()
                os.symlink(found, dst)
    return (len(missing) == 0 and bool(tgts)), missing

_ig.image_model_status = lambda key, model_dir=None: (lambda ok_a, ms: {"key": key, "dir": str(Path(model_dir or _cfg.MODEL_DIR) / key), "downloaded": ok_a, "missing": ms, "gb": 0.0, "approx_gb": 0.0})(*_sync_image_preset(key, str(model_dir or _cfg.MODEL_DIR)))
_ig.link_image_model = lambda key, model_dir=None, comfy_dir=None: (_sync_image_preset(key, str(model_dir or _cfg.MODEL_DIR)), {})[1]
_ig.ensure_public_url = lambda port=8188, prefer="local": (f"http://127.0.0.1:{int(port)}", "local")
if not shutil.which("llama-server"): install_llama_server()

def _is_aux_gguf(p: Path) -> bool:
    nm = p.name.lower()
    return ("mmproj" in nm or "eagle3" in nm or nm.startswith(("imatrix", "mtp-", "mtp_", "draft-", "draft_")) or nm.endswith(("-imatrix.gguf", "-draft.gguf", "_draft.gguf")) or any(x in nm for x in ("-mtp-draft", "-draft-", "_draft-", ".draft.")))

_SPEC_CACHE: dict[str, dict] = {}
_SPEC_LOCK = threading.Lock()

def _measure_live_mem() -> dict:
    ram_a = ram_tot = 0.0
    if Path("/proc/meminfo").is_file():
        for ln in Path("/proc/meminfo").read_text(errors="ignore").splitlines():
            if ln.startswith("MemTotal:"): ram_tot = int(ln.split()[1]) / (1024**2)
            elif ln.startswith("MemAvailable:"): ram_a = int(ln.split()[1]) / (1024**2)
    eff_ram = max(ram_a, ram_tot - 2.6) if ram_tot > 0 else ram_a
    smi = shutil.which("nvidia-smi") or ("/usr/bin/nvidia-smi" if Path("/usr/bin/nvidia-smi").exists() else None)
    if not smi: return {"has_gpu": False, "eff_vram_gb": 0.0, "free_vram_gb": 0.0, "max_single_gb": 0.0, "ram_avail_gb": round(eff_ram, 2)}
    reclaim, per_eff, per_free = {}, [], []
    with contextlib.suppress(Exception):
        for ln in subprocess.check_output([smi, "--query-compute-apps=gpu_uuid,process_name,used_memory", "--format=csv,noheader,nounits"], text=True, stderr=subprocess.DEVNULL, timeout=3).splitlines():
            pts = [x.strip() for x in ln.split(",")]
            if len(pts) >= 3 and any(w in pts[1].lower() for w in ("llama-server", "python", "comfy")):
                reclaim[pts[0]] = reclaim.get(pts[0], 0.0) + float(pts[2] or 0) / 1024.0
        for ln in subprocess.check_output([smi, "--query-gpu=uuid,memory.total,memory.free", "--format=csv,noheader,nounits"], text=True, stderr=subprocess.DEVNULL, timeout=3).splitlines():
            pts = [x.strip() for x in ln.split(",")]
            if len(pts) >= 3:
                tot_g, fr_g = float(pts[1]) / 1024.0, float(pts[2]) / 1024.0
                per_free.append(fr_g)
                per_eff.append(max(0.0, min(tot_g, fr_g + reclaim.get(pts[0], 0.0)) - 0.45))
    return {"has_gpu": bool(per_eff), "eff_vram_gb": round(sum(per_eff), 2), "free_vram_gb": round(sum(per_free), 2), "max_single_gb": round(max(per_eff, default=0.0), 2), "ram_avail_gb": round(eff_ram, 2)}

def _probe_spec(key: str, model_dir: str, allow_net: bool = False) -> dict | None:
    cfg = _cfg.MODEL_CATALOG.get(key)
    if not cfg: return None
    sub, kind = Path(model_dir) / key, str(cfg.get("kind") or "text")
    if sub.is_dir():
        if kind == "lora":
            sfs = [p for p in sub.rglob("*.safetensors") if p.is_file() and not p.name.endswith(".part") and p.stat().st_size > 0]
            if sfs:
                sp = {"kind": "lora", "total_bytes": sum(p.stat().st_size for p in sfs)}
                with _SPEC_LOCK: _SPEC_CACHE[key] = sp
                return sp
        elif kind == "image":
            tgts = cfg.get("targets") or {}
            res = [(role, _resolve_local_target(sub, r)) for r, role in tgts.items()]
            if res and all(p is not None for _, p in res):
                sb = {"diffusion_models": 0, "text_encoders": 0, "vae": 0, "other": 0}
                for role, p in res: sb[role if role in sb else "other"] += p.stat().st_size  # type: ignore[union-attr]
                sp = {"kind": "image", "total_bytes": sum(sb.values()), "diff_gb": sb["diffusion_models"] / 1024**3, "te_gb": sb["text_encoders"] / 1024**3, "vae_gb": sb["vae"] / 1024**3}
                with _SPEC_LOCK: _SPEC_CACHE[key] = sp
                return sp
        else:
            ggufs = [p for p in sorted(sub.rglob("*.gguf")) if p.is_file() and not p.name.endswith(".part")]
            mains = [p for p in ggufs if not _is_aux_gguf(p)]
            if mains:
                sp = {"kind": "text", "total_bytes": sum(p.stat().st_size for p in ggufs), "main_gb": sum(p.stat().st_size for p in mains) / 1024**3, "mm_gb": sum(p.stat().st_size for p in ggufs if "mmproj" in p.name.lower()) / 1024**3, "n_experts": (2 if any(t in key for t in ("8x7b", "a2b", "a3b", "a4b", "bolt-74b", "gpt-oss")) else 0)}
                with _SPEC_LOCK: _SPEC_CACHE[key] = sp
                return sp
    with _SPEC_LOCK: hit = _SPEC_CACHE.get(key)
    if hit is not None or not allow_net: return hit
    try:
        from huggingface_hub import HfApi
        tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        api, srcs, inc, exc = HfApi(token=tok or False), cfg.get("sources") or {}, list(cfg.get("include") or ["*"]), list(cfg.get("exclude") or [])
        by_r: dict[str, list[str]] = {}
        for pat in inc: by_r.setdefault(srcs.get(pat, cfg["repo_id"]), []).append(pat)
        matched = [(rid, sib.rfilename, int(getattr(sib, "size", 0) or 0)) for rid, pats in by_r.items() for sib in (api.repo_info(repo_id=rid, repo_type="model", files_metadata=True).siblings or []) if any(fnmatch.fnmatch(sib.rfilename, p) or fnmatch.fnmatch(Path(sib.rfilename).name, p) for p in pats) and not (exc and any(fnmatch.fnmatch(sib.rfilename, p) or fnmatch.fnmatch(Path(sib.rfilename).name, p) for p in exc))]
        if not matched: raise FileNotFoundError(key)
        if kind == "lora": sp = {"kind": "lora", "total_bytes": matched[0][2]}
        elif kind == "image":
            tgts = cfg.get("targets") or {}
            sb = {"diffusion_models": 0, "text_encoders": 0, "vae": 0, "other": 0}
            for _r, fn, sz in matched:
                role = tgts.get(fn) or next((v for kt, v in tgts.items() if Path(kt).name.lower() == Path(fn).name.lower()), "other")
                sb[role if role in sb else "other"] += sz
            sp = {"kind": "image", "total_bytes": sum(sz for _, _, sz in matched), "diff_gb": sb["diffusion_models"] / 1024**3, "te_gb": sb["text_encoders"] / 1024**3, "vae_gb": sb["vae"] / 1024**3}
        else:
            mains_r = [sz for _, fn, sz in matched if fn.lower().endswith(".gguf") and not _is_aux_gguf(Path(fn))] if matched else []
            sp = {"kind": "text", "total_bytes": sum(sz for _, _, sz in matched), "main_gb": sum(mains_r) / 1024**3, "mm_gb": sum(sz for _, fn, sz in matched if "mmproj" in fn.lower()) / 1024**3, "n_experts": (2 if any(t in key for t in ("8x7b", "a2b", "a3b", "a4b", "bolt-74b", "gpt-oss")) else 0)}
    except Exception:
        apx = float(cfg.get("approx_gb") or 8.0)
        sp = {"kind": kind, "total_bytes": int(apx * 1024**3), "main_gb": apx, "mm_gb": 0.0, "diff_gb": apx * 0.6, "te_gb": apx * 0.35, "vae_gb": 0.3, "n_experts": 0}
    with _SPEC_LOCK: _SPEC_CACHE[key] = sp
    return sp

def _eval_real_vram_fit(key: str, model_dir: str, mem: dict, kv_q8: bool) -> dict:
    sp = _probe_spec(key, model_dir, allow_net=False)
    if sp is None: return {"fit": "probing", "fit_label": "⏳ กำลังวัดขนาดจริง...", "bytes": 0}
    ram_b, tot_b = max(0.0, float(mem["ram_avail_gb"]) - 1.2), int(sp["total_bytes"])
    if sp["kind"] == "lora": return {"fit": "ok", "fit_label": f"🎮 พอดี VRAM (+LoRA {tot_b/1024**3:.2f}GB)", "bytes": tot_b}
    if sp["kind"] == "image":
        dg, tg, vg = float(sp["diff_gb"]), float(sp["te_gb"]), float(sp["vae_gb"])
        peak, sg = max(dg, tg) + vg + 1.15, float(mem["max_single_gb"])
        if not mem["has_gpu"]: return {"fit": "warn", "fit_label": f"🖥️ CPU ({tot_b/1024**3:.1f}GB)", "bytes": tot_b}
        if peak <= sg: return {"fit": "ok", "fit_label": f"🎮 พอดี VRAM ({peak:.1f}/{sg:.1f}GB)", "bytes": tot_b}
        if max(dg, tg) <= sg + ram_b: return {"fit": "warn", "fit_label": f"⚠️ LowVRAM สลับ RAM ({peak:.1f}/{sg:.1f}GB)", "bytes": tot_b}
        return {"fit": "over", "fit_label": "🚫 เกิน VRAM+RAM", "bytes": tot_b}
    mg, mmg, nexp = float(sp["main_gb"]), float(sp["mm_gb"]), int(sp.get("n_experts", 0))
    need, ev = mg + max(0.25, mg * (0.035 if kv_q8 else 0.068)) + 0.55 + mmg, float(mem["eff_vram_gb"])
    if not mem["has_gpu"]: return {"fit": "warn" if need <= ram_b else "over", "fit_label": f"🖥️ RAM ({need:.1f}/{ram_b:.1f}GB)", "bytes": tot_b}
    if need <= ev: return {"fit": "ok", "fit_label": f"🎮 พอดี VRAM ({need:.1f}/{ev:.1f}GB)", "bytes": tot_b}
    off = need - ev
    if nexp > 1 and (mg * 0.30 + 0.8) <= ev and off <= ram_b: return {"fit": "moe", "fit_label": f"⚡ พอดี GPU+MoE (+RAM {off:.1f}GB)", "bytes": tot_b}
    if off <= ram_b: return {"fit": "warn", "fit_label": f"⚠️ แบ่ง GPU+RAM (+RAM {off:.1f}GB)", "bytes": tot_b}
    return {"fit": "over", "fit_label": "🚫 เกิน VRAM+RAM", "bytes": tot_b}

threading.Thread(target=lambda: list(concurrent.futures.ThreadPoolExecutor(max_workers=8).map(lambda k: _probe_spec(k, _cfg.MODEL_DIR, True), list(_cfg.ALL_PRESET_KEYS))), daemon=True).start()

_DL_PROG = {"active": False, "pct": 0.0, "done": "0 B", "total": "0 B", "rate": "0 B/s", "eta": "-", "files": "0/0", "file": "", "repo": ""}
_DL_CANCEL = threading.Event()

def _fast_cdn_download(repo_id: str, dest: Path, include=None, exclude=None, single_first: bool = False) -> None:
    from huggingface_hub import HfApi, hf_hub_url
    _DL_CANCEL.clear(); dest.mkdir(parents=True, exist_ok=True)
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    info = HfApi(token=tok or False).repo_info(repo_id=repo_id, repo_type="model", files_metadata=True)
    inc, exc = (list(include) if include else ["*"]), (list(exclude) if exclude else [])
    sel = [(s.rfilename, int(getattr(s, "size", 0) or 0)) for s in (info.siblings or []) if any(fnmatch.fnmatch(s.rfilename, p) or fnmatch.fnmatch(Path(s.rfilename).name, p) for p in inc) and not (exc and any(fnmatch.fnmatch(s.rfilename, p) or fnmatch.fnmatch(Path(s.rfilename).name, p) for p in exc))]
    if not sel: raise FileNotFoundError(f"ไม่พบไฟล์ {inc} ใน {repo_id}")
    if single_first and len(sel) > 1: sel = [sel[0]]
    tot_all, done_prev = sum(sz for _, sz in sel), 0
    _DL_PROG.update({"active": True, "repo": repo_id, "pct": 0.0, "files": f"0/{len(sel)}"})
    try:
        for idx, (rel, fsize) in enumerate(sel, 1):
            if _DL_CANCEL.is_set(): raise RuntimeError("ยกเลิกการดาวน์โหลดแล้ว")
            out_f = dest / rel; out_f.parent.mkdir(parents=True, exist_ok=True)
            if out_f.is_file() and fsize > 0 and out_f.stat().st_size == fsize: done_prev += fsize; continue
            part_f = out_f.with_suffix(out_f.suffix + ".part")
            res_at = part_f.stat().st_size if part_f.is_file() else 0
            hdrs = {"User-Agent": "strata-studio/2.0", **({"Authorization": f"Bearer {tok}"} if tok else {}), **({"Range": f"bytes={res_at}-"} if res_at > 0 else {})}
            with urllib.request.urlopen(urllib.request.Request(hf_hub_url(repo_id=repo_id, filename=rel, repo_type="model"), headers=hdrs), timeout=60) as resp:
                mode = "ab" if (res_at > 0 and getattr(resp, "status", 200) == 206) else "wb"
                if mode == "wb": res_at = 0
                cur, lt, lb, rema = res_at, time.perf_counter(), done_prev + res_at, 0.0
                with part_f.open(mode) as fh:
                    while True:
                        if _DL_CANCEL.is_set(): raise RuntimeError("ยกเลิกแล้ว (เก็บ .part ไว้โหลดต่อได้)")
                        buf = resp.read(4 * 1024 * 1024)
                        if not buf: break
                        fh.write(buf); cur += len(buf); now = time.perf_counter()
                        if (now - lt) >= 0.35:
                            td = done_prev + cur; inst = (td - lb) / max(0.05, now - lt)
                            rema = inst if rema <= 0 else (0.35 * inst + 0.65 * rema)
                            eta = int(max(0, tot_all - td) / rema) if rema > 1024 else 0
                            _DL_PROG.update({"active": True, "pct": min(100.0, td / max(1, tot_all) * 100.0), "done": _fmt_bytes(td), "total": _fmt_bytes(tot_all), "rate": _fmt_rate(rema), "eta": f"{eta // 60}m {eta % 60:02d}s" if eta >= 60 else f"{eta}s", "files": f"{idx}/{len(sel)}", "file": Path(rel).name, "repo": repo_id})
                            lt, lb = now, td
            part_f.replace(out_f); done_prev += fsize or out_f.stat().st_size
    finally:
        _DL_PROG["active"] = False

def _smart_install_any(key: str, model_dir: str) -> None:
    cfg, sub = _cfg.MODEL_CATALOG[key], Path(model_dir) / key
    kind, srcs, by_r = str(cfg.get("kind") or "text"), cfg.get("sources") or {}, {}
    for pat in (cfg.get("include") or ["*"]): by_r.setdefault(srcs.get(pat, cfg["repo_id"]), []).append(pat)
    for rid, pats in by_r.items(): _fast_cdn_download(rid, sub, include=pats, exclude=cfg.get("exclude"), single_first=(kind == "lora"))
    if kind == "image":
        _sync_image_preset(key, model_dir)
        bl = next((Path(r).name for r, role in (cfg.get("targets") or {}).items() if role == "loras"), "")
        _write_sync_state(active_model=key, **({"active_lora": bl, "lora_strength": 1.0} if bl else {}))
        _ST["live_step"] = f"ซิงก์ {key} เข้าเซลล์ 3 อัตโนมัติแล้ว"
    elif kind == "lora":
        ok_l, lf = _sync_lora_preset(key, model_dir)
        if ok_l and lf: _write_sync_state(active_lora=lf, lora_strength=float(cfg.get("default_strength", 1.0))); _ST["live_step"] = f"ซิงก์ Uncensored LoRA ({lf}) เข้าเซลล์ 3 แล้ว"
    elif str(_cfg.SPEC_STATE.get("mode", "auto")).lower() == "auto":
        dk = _pick_auto_draft_key(key)
        if dk and dk in _cfg.DRAFT_CATALOG:
            dc, dsub = _cfg.DRAFT_CATALOG[dk], Path(_cfg.DRAFT_DIR) / dk
            if not _find_draft_file(dsub):
                _ST["live_step"] = f"✨ Auto: กำลังโหลด MODEL Draft ({dk}) อัตโนมัติ..."
                _fast_cdn_download(dc["repo_id"], dsub, include=dc.get("include") or ["*.gguf"], exclude=dc.get("exclude"), single_first=True)
            if (fp := _find_draft_file(dsub)):
                _cfg.SPEC_STATE["draft_model"] = str(fp); _cfg.SPEC_STATE["selected_draft_key"] = dk
                _ST["live_step"] = f"✨ Auto: โหลดพร้อม MODEL Draft ({dk}) เรียบร้อย"
    _invalidate_dir_cache()
    with _SPEC_LOCK: _SPEC_CACHE.pop(key, None)

_PLAT, _MDIR, _PORT = _detect_plat(), _cfg.MODEL_DIR, int(_cfg.DEFAULT_PORT)
_TPLS = getattr(_cfg, "SYSTEM_PROMPT_TEMPLATES", {
    "assistant": {"label": "🤖 ผู้ช่วยอัจฉริยะ", "prompt": "คุณคือผู้ช่วย AI อัจฉริยะที่ตอบชัดเจนและตรงประเด็น", "temp": 0.7, "min_p": 0.05, "repeat_penalty": 1.08, "dry": 0.0, "max_tokens": 1024},
    "rp_immersive": {"label": "🎭 Roleplay สวมบทบาทสมจริง", "prompt": "คุณคือนักเล่าเรื่องสวมบทบาทสมจริง บรรยายอารมณ์ ท่าทาง บทพูด และบรรยากาศอย่างมีชีวิตชีวา", "temp": 0.95, "min_p": 0.05, "repeat_penalty": 1.05, "dry": 0.8, "max_tokens": 1536},
})

def _list_ready_loras() -> list[dict]:
    ldir = Path(_cfg.COMFY_DIR) / "models" / "loras"; ldir.mkdir(parents=True, exist_ok=True)
    for k in _cfg.ALL_PRESET_KEYS:
        if (_cfg.MODEL_CATALOG.get(k) or {}).get("kind") == "lora": _sync_lora_preset(k, _MDIR)
    return [{"file": p.name, "label": f"🔥 {p.name}"} for p in sorted(ldir.iterdir()) if (p.is_file() or p.is_symlink()) and p.exists() and p.suffix.lower() in (".safetensors", ".pt", ".gguf")]

def _read_comfy_log_tail(n: int = 12) -> str:
    lp = Path(_cfg.COMFY_DIR) / "comfyui.log"
    return "\n".join(lp.read_text(encoding="utf-8", errors="ignore").splitlines()[-n:]) if lp.is_file() else ""

def _run_quick_comfy(key: str, wf: dict, lora_name: str = "", lora_str: float = 1.0) -> dict:
    import requests
    _free_single_gpu_for_comfy(); _ig.ensure_comfyui(pull=False)
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2.0)
    except Exception:
        env_c = {k: v for k, v in os.environ.items() if k != "PYTHONHOME"}
        env_c["LD_LIBRARY_PATH"] = ":".join(p for p in env_c.get("LD_LIBRARY_PATH", "").split(":") if p and not p.startswith("/nix/store/"))
        has_g = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
        subprocess.Popen([sys.executable, str(Path(_cfg.COMFY_DIR) / "main.py"), "--listen", "0.0.0.0", "--port", "8188", "--preview-method", "none", "--lowvram" if has_g else "--cpu"], cwd=str(_cfg.COMFY_DIR), stdout=(Path(_cfg.COMFY_DIR) / "comfyui.log").open("w"), stderr=subprocess.STDOUT, env=env_c)
        for _ in range(90):
            with contextlib.suppress(Exception):
                if urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2.0).status == 200: break
            time.sleep(1.2)
    tgts = (_cfg.MODEL_CATALOG[key] or {}).get("targets") or {}
    unet = next((Path(r).name for r, role in tgts.items() if role == "diffusion_models"), "")
    clip = next((Path(r).name for r, role in tgts.items() if role == "text_encoders"), "")
    vae = next((Path(r).name for r, role in tgts.items() if role == "vae"), "")
    def_lora = next((Path(r).name for r, role in tgts.items() if role == "loras"), "")
    use_l = lora_name if lora_name != "__AUTO__" else def_lora
    active_l = bool(use_l) and abs(lora_str) > 1e-6 and (Path(_cfg.COMFY_DIR) / "models" / "loras" / use_l).is_file()
    hk = key.lower()
    ctype = "lumina2" if "z-image" in hk else ("chroma" if "chroma" in hk else "qwen_image")
    c_cls = "CLIPLoaderGGUF" if clip.lower().endswith(".gguf") else "CLIPLoader"
    c_in = {"clip_name": clip, "type": ctype} if c_cls == "CLIPLoaderGGUF" else {"clip_name": clip, "type": ctype, "device": "default" if _has_gpu else "cpu"}
    seed = int(wf["seed"]) if int(wf.get("seed", -1)) >= 0 else random.randint(1, 2**31 - 1)
    w, h, steps, cfg_v = int(wf["width"]), int(wf["height"]), int(wf["steps"]), float(wf["cfg"])
    graph = {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": unet}},
        "2": {"class_type": c_cls, "inputs": c_in},
        "3": {"class_type": "CLIPTextEncode", "inputs": {"text": str(wf["pos"]), "clip": ["2", 0]}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"text": str(wf["neg"]), "clip": ["2", 0]}},
        "5": {"class_type": "EmptySD3LatentImage", "inputs": {"width": w, "height": h, "batch_size": 1}},
        "6": {"class_type": "KSampler", "inputs": {"model": ["10" if active_l else "1", 0], "positive": ["3", 0], "negative": ["4", 0], "latent_image": ["5", 0], "seed": seed, "steps": steps, "cfg": cfg_v, "sampler_name": str(wf["sampler"]), "scheduler": str(wf["scheduler"]), "denoise": 1.0}},
        "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["8", 0]}},
        "8": {"class_type": "VAELoader", "inputs": {"vae_name": vae}},
        "9": {"class_type": "SaveImage", "inputs": {"images": ["7", 0], "filename_prefix": "strata"}},
    }
    if active_l: graph["10"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": use_l, "strength_model": float(lora_str)}}
    t0 = time.perf_counter()
    rp = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": graph, "client_id": "strata-tab2"}, timeout=30)
    if rp.status_code >= 400: raise RuntimeError(f"ComfyUI ปฏิเสธ workflow: {rp.text[:300]}")
    pid = rp.json().get("prompt_id", "")
    for _ in range(600):
        el = int(time.perf_counter() - t0)
        try: hist = (requests.get(f"http://127.0.0.1:8188/history/{pid}", timeout=8).json() or {}).get(pid) or {}
        except Exception: hist = {}
        if (hist.get("status") or {}).get("status_str") == "error":
            raise RuntimeError(f"ComfyUI error: {json.dumps((hist.get('status') or {}).get('messages', []), ensure_ascii=False)[:350]}")
        imgs = [im for no in (hist.get("outputs") or {}).values() for im in (no.get("images") or []) if im.get("filename")]
        if imgs:
            im0 = imgs[0]; out_d = Path(_cfg.STRATA_HOME) / "outputs"; out_d.mkdir(parents=True, exist_ok=True)
            dst = out_d / f"{key}_{int(time.time())}_{im0['filename']}"
            dst.write_bytes(requests.get("http://127.0.0.1:8188/view", params={"filename": im0['filename'], "subfolder": im0.get("subfolder", ""), "type": im0.get("type", "output")}, timeout=60).content)
            return {"path": str(dst), "seconds": round(time.perf_counter() - t0, 1), "w": w, "h": h}
        tail = _read_comfy_log_tail(5)
        if "OutOfMemoryError" in tail or "CUDA out of memory" in tail: raise RuntimeError(f"VRAM ไม่พอ:\n{tail}")
        ph = re.findall(r"(\d+%\|.*?\|\s*\d+/\d+)", tail)
        _ST["img_stat"] = f"⏳ กำลังวาด {key} [{ph[-1] if ph else f'{steps} steps'}] ({w}×{h} • CFG {cfg_v} • {el}s)..."
        time.sleep(1.0)
    raise TimeoutError(f"ComfyUI เกิน 10 นาที:\n{_read_comfy_log_tail(10)}")

def _snap() -> dict:
    cp = Path(_MDIR) if Path(_MDIR).exists() else Path(_PLAT["home"])
    du, hw, live_mem = shutil.disk_usage(cp), check_gpu_and_recommend(verbose=False), _measure_live_mem()
    use_q8 = bool(_cfg.SPEC_STATE.get("kv_q8", True))
    on, mid = _check_server_online(_PORT)
    net, base, models, ready_imgs = net_bandwidth(), Path(_MDIR), [], []
    sync_st = _read_sync_state()
    for k in _cfg.ALL_PRESET_KEYS:
        c, sub = _cfg.MODEL_CATALOG[k], base / k
        ex = sub.is_dir()
        st = _dir_stats(sub) if ex else {"bytes": 0, "gguf": 0}
        kind, lfn = str(c.get("kind") or "text"), ""
        if kind == "image":
            rdy = _sync_image_preset(k, _MDIR)[0] if ex else False
            if rdy: ready_imgs.append(k)
        elif kind == "lora":
            rdy, lfn = _sync_lora_preset(k, _MDIR) if ex else (False, "")
        else:
            rdy = int(st["gguf"]) > 0
        vr = _eval_real_vram_fit(k, _MDIR, live_mem, use_q8)
        rb = int(st["bytes"]) if rdy else int(vr["bytes"])
        models.append({"key": k, "kind": kind, "desc": c["desc"], "repo": c["repo_id"], "installed": rdy, "size_txt": _fmt_bytes(rb) if rb > 0 else "กำลังเช็กขนาด...", "tags": list(c.get("tags") or []), "fit": vr["fit"], "fit_label": vr["fit_label"], "can_del": ex, "lora_file": lfn or str(c.get("lora_file") or ""), "auto_draft": (_pick_auto_draft_key(k) or "") if kind == "text" else ""})
    if ready_imgs and (not sync_st.get("active_model") or sync_st.get("active_model") not in ready_imgs): sync_st = _write_sync_state(active_model=ready_imgs[0])
    elif not ready_imgs and sync_st.get("active_model"): sync_st = _write_sync_state(active_model="")
    drafts = [{"key": dk, "label": f"{'✅ ' if _find_draft_file(Path(_cfg.DRAFT_DIR) / dk) else '⬜ '}{dv['desc']}"} for dk, dv in _cfg.DRAFT_CATALOG.items()]
    gpu_lbl = f"🎮 {hw['gpu_name']} (สลับใช้ได้ {live_mem['eff_vram_gb']:.1f}/{hw['vram_total_gb']:.1f}GB • ว่างสด {live_mem['free_vram_gb']:.1f}GB)" if hw["has_gpu"] else f"🖥️ CPU (RAM {live_mem['ram_avail_gb']:.1f}GB)"
    return {"online": on, "active_model": mid if on else "Offline", "port": _PORT, "platform": _PLAT["label"], "spec": {"mode": _cfg.SPEC_STATE.get("mode", "auto"), "draft_max": int(_cfg.SPEC_STATE.get("draft_max", 0)), "kv_q8": use_q8, "selected_draft_key": str(_cfg.SPEC_STATE.get("selected_draft_key", ""))} , "sync": sync_st, "ready_loras": _list_ready_loras(), "rag_chunks": len(GLOBAL_RAG.chunks), "gpu": gpu_lbl, "disk": {"free_gb": round(du.free / 1024**3, 2), "total_gb": round(du.total / 1024**3, 2), "pct": round(du.used / max(1, du.total) * 100, 1)}, "net": f"⬇ {_fmt_rate(net['down_bps'])} · ⬆ {_fmt_rate(net['up_bps'])}", "job": _ST["job"], "live_step": _ST["live_step"], "notice": _ST["notice"], "kind": _ST["kind"], "progress": dict(_DL_PROG) if _DL_PROG.get("active") else None, "img_stat": _ST["img_stat"], "last_img": _ST["last_img"], "models": models, "drafts": drafts}

def _bg_run(lbl: str, fn) -> None:
    if _ST["job"]: _ST.update({"notice": f"⏳ กำลังทำ '{_ST['job']}' อยู่ — กด ⏹️ ยกเลิกงาน ก่อนครับ", "kind": "warn"}); return
    def _w():
        _ST.update({"job": lbl, "live_step": "", "notice": "", "kind": "info"})
        try:
            fn(); _ST.update({"notice": f"✅ สำเร็จ: {lbl}" + (f" — {_ST['live_step']}" if _ST['live_step'] else ""), "kind": "ok"})
        except Exception as e:
            _ST.update({"notice": f"❌ {e}", "kind": "err"})
        finally:
            _ST["job"] = ""; _ST["live_step"] = ""; _invalidate_dir_cache(); _invalidate_server_cache(_PORT)
    threading.Thread(target=_w, daemon=True).start()

def _handle_action(act: str, payload: dict | None = None) -> dict:
    d = payload or {}
    try:
        if act == "poll": return _snap()
        if act == "cancel":
            _DL_CANCEL.set(); _DL_PROG["active"] = False
            with contextlib.suppress(Exception): urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/interrupt", data=b"{}", headers={"Content-Type": "application/json"}), timeout=1.5)
            _ST.update({"job": "", "live_step": "", "notice": "⏹️ ยกเลิกงานค้างเรียบร้อย", "kind": "warn"}); return _snap()
        if act == "auto_workflow":
            rep_txt = str((d.get("reply_to") or {}).get("text", "")).strip() if isinstance(d.get("reply_to"), dict) else ""
            src_txt = _merge_reply_and_prompt(rep_txt, str(d.get("prompt", "")).strip())
            if d.get("from_last_chat") and not src_txt and _ST["history"]:
                src_txt = str(_ST["history"][-1].get("content", ""))[:350]
            st_cur = _read_sync_state()
            mkey = str(d.get("key") or st_cur.get("active_model") or "qwen-image-21-turbo")
            lnm = str(d.get("lora_name", st_cur.get("active_lora", "")))
            lstr = float(d.get("lora_strength", st_cur.get("lora_strength", 1.0)))
            wf = _auto_build_workflow(src_txt, mkey, lnm, lstr, use_llm=bool(d.get("use_llm")), port=_PORT)
            _write_sync_state(active_model=mkey if mkey in _cfg.MODEL_CATALOG else st_cur.get("active_model", ""), active_lora=lnm, lora_strength=lstr, workflow=wf)
            return {"ok": True, "workflow": wf, "from_chat": src_txt}
        if act == "sync_comfy_sel":
            upd = {k: d[k] for k in ("active_model", "active_lora", "lora_strength") if k in d}
            if "prompt" in d:
                st_cur = {**_read_sync_state(), **upd}
                upd["workflow"] = _auto_build_workflow(str(d["prompt"]), str(st_cur.get("active_model") or "qwen-image-21-turbo"), str(st_cur.get("active_lora") or ""), float(st_cur.get("lora_strength", 1.0)))
            if upd: _write_sync_state(**upd)
            return _snap()
        if act == "link_image":
            key = str(d.get("key", ""))
            if key in _cfg.MODEL_CATALOG:
                if (_cfg.MODEL_CATALOG[key] or {}).get("kind") == "lora":
                    ok_l, lf = _sync_lora_preset(key, _MDIR)
                    if ok_l:
                        ds = float((_cfg.MODEL_CATALOG[key] or {}).get("default_strength", 1.0))
                        _write_sync_state(active_lora=lf, lora_strength=ds)
                        _ST.update({"notice": f"🔥🧩 เลือก Uncensored LoRA '{lf}' (×{ds:.2f}) ซิงก์เข้าเซลล์ 3 แล้ว!", "kind": "ok"})
                else:
                    ok_a, ms = _sync_image_preset(key, _MDIR)
                    if ok_a:
                        bl = next((Path(r).name for r, role in ((_cfg.MODEL_CATALOG[key] or {}).get("targets") or {}).items() if role == "loras"), "")
                        _write_sync_state(active_model=key, **({"active_lora": bl, "lora_strength": 1.0} if bl else {}))
                        _ST.update({"notice": f"🎨 ซิงก์โมเดลภาพ '{key}' เข้าเซลล์ 3 แล้ว!", "kind": "ok"})
                    else:
                        _ST.update({"notice": f"🔒 ยังขาดไฟล์: {', '.join(ms)}", "kind": "warn"})
            return _snap()
        if act == "set_spec":
            _cfg.SPEC_STATE.update({"mode": str(d.get("mode", "auto")), "_user_set_mode": True, "draft_max": int(d.get("draft_max", 0)), "kv_q8": bool(d.get("kv_q8", True))})
            if d.get("restart"):
                on, mid = _check_server_online(_PORT, ttl=0.0)
                if on and (Path(_MDIR) / mid).is_dir():
                    if _cfg.SPEC_STATE["mode"] in ("auto", "draft"): _cfg.SPEC_STATE["draft_model"] = _resolve_draft_for_run(mid)
                    _bg_run(f"รีสตาร์ทโหมด {_cfg.SPEC_STATE['mode']}", lambda: (_free_comfy_vram_for_llm(), start_strata_server(model_dir=str(Path(_MDIR) / mid), port=_PORT, restart=True)))
            return _snap()
        if act == "dl_draft":
            dk = str(d.get("key", ""))
            if dk in _cfg.DRAFT_CATALOG:
                dc = _cfg.DRAFT_CATALOG[dk]
                def _jd():
                    _fast_cdn_download(dc["repo_id"], Path(_cfg.DRAFT_DIR) / dk, include=dc.get("include") or ["*.gguf"], exclude=dc.get("exclude"), single_first=True)
                    if (fp := _find_draft_file(Path(_cfg.DRAFT_DIR) / dk)): _cfg.SPEC_STATE["draft_model"] = str(fp); _cfg.SPEC_STATE["selected_draft_key"] = dk
                _bg_run(f"โหลด Draft {dk}", _jd)
            return _snap()
        if act == "model_op":
            op, key = str(d.get("op", "")), str(d.get("key", ""))
            if op == "download" and key in _cfg.MODEL_CATALOG: _bg_run(f"โหลด {key}", lambda: _smart_install_any(key, _MDIR))
            elif op == "run" and key in _cfg.MODEL_CATALOG:
                if not ((Path(_MDIR) / key).is_dir() and _dir_stats(Path(_MDIR) / key)["gguf"] > 0):
                    _ST.update({"notice": f"⚠️ กด 📥 โหลด '{key}' ก่อนครับ", "kind": "warn"}); return _snap()
                if str(_cfg.SPEC_STATE.get("mode", "auto")).lower() in ("auto", "draft"): _cfg.SPEC_STATE["draft_model"] = _resolve_draft_for_run(key)
                _bg_run(f"รันโมเดล {key}", lambda: (_free_comfy_vram_for_llm(), start_strata_server(model_dir=str(Path(_MDIR) / key), port=_PORT, restart=True)))
            elif op == "delete" and key in _cfg.MODEL_CATALOG:
                def _jdel():
                    _dl.remove_model(key, model_dir=_MDIR)
                    for _sd in ("diffusion_models", "text_encoders", "vae", "loras", "clip", "unet"):
                        _pd = Path(_cfg.COMFY_DIR) / "models" / _sd
                        if _pd.is_dir():
                            for _sl in _pd.iterdir():
                                if _sl.is_symlink() and not _sl.exists(): _sl.unlink(missing_ok=True)
                    _write_sync_state()
                    with _SPEC_LOCK: _SPEC_CACHE.pop(key, None)
                    _probe_spec(key, _MDIR, allow_net=True)
                _bg_run(f"ลบ {key}", _jdel)
            elif op == "stop_server": stop_strata_server(); _invalidate_server_cache(_PORT); _ST.update({"notice": "🛑 หยุด llama-server แล้ว", "kind": "info"})
            elif op == "clean_cache": _bg_run("ล้างแคช", lambda: _dl.clean_download_cache(model_dir=_MDIR))
            elif op == "delete_all": _bg_run("ลบทั้งหมด", lambda: (_dl.remove_model("all", model_dir=_MDIR, stop_server=True), _write_sync_state(active_model="", active_lora="")))
            return _snap()
        if act == "chat":
            on, mid = _check_server_online(_PORT, ttl=0.3)
            if not on: return {"ok": False, "error": f"Server แชทพอร์ต {_PORT} ยังไม่เปิด — โหลดโมเดลข้อความแล้วกด 🚀 รัน & แชท ในแท็บ 1 (หากต้องการวาดภาพจากข้อความที่รีพาย กดปุ่มสีส้ม 🖼️ สร้างภาพ ด้านล่างได้เลยครับ)"}
            hist = _ST["history"]
            rep_obj = d.get("reply_to") if isinstance(d.get("reply_to"), dict) else None
            rep_txt = str((rep_obj or {}).get("text", "")).strip()
            rep_who = str((rep_obj or {}).get("who", "ข้อความก่อนหน้า")).strip()
            if d.get("regen"):
                if len(hist) < 2: return {"ok": False, "error": "ยังไม่มีคำตอบให้สุ่มใหม่"}
                hist.pop(); q_send = hist.pop()["content"]
            else:
                q_raw = str(d.get("prompt", "")).strip()
                if not q_raw and rep_txt: q_raw = "ช่วยขยายความหรือเล่าต่อจากข้อความนี้ให้ละเอียดขึ้น"
                if not q_raw: return {"ok": False, "error": "พิมพ์ข้อความก่อนส่งครับ"}
                q_send = f"[อ้างอิงข้อความที่รีพาย ({rep_who})]:\n\"{rep_txt[:600]}\"\n\n[คำถาม/ข้อความตอบกลับ]: {q_raw}" if rep_txt else q_raw
            aug_q, docs = GLOBAL_RAG.build_augmented_prompt(q_send, top_k=3) if (d.get("use_rag") and GLOBAL_RAG.chunks) else (q_send, [])
            extra = {"min_p": float(d.get("min_p", 0.05)), "repeat_penalty": float(d.get("repeat_penalty", 1.08)), **({"dry_multiplier": float(d["dry"])} if float(d.get("dry", 0.0)) > 0 else {})}
            t0 = time.perf_counter()
            resp = get_openai_client(port=_PORT).chat.completions.create(model=mid, messages=[{"role": "system", "content": str(d.get("system", ""))}, *hist, {"role": "user", "content": aug_q}], temperature=float(d.get("temp", 0.7)), max_tokens=int(d.get("max_tokens", 1024)), extra_body=extra)
            el = max(0.01, time.perf_counter() - t0)
            reply = (resp.choices[0].message.content or "") if resp.choices else ""
            toks = getattr(resp.usage, "completion_tokens", None) or max(1, len(reply) // 4)
            hist.extend([{"role": "user", "content": q_send}, {"role": "assistant", "content": reply}])
            while len(hist) > 20: hist.pop(0)
            if d.get("auto_wf", True):
                st_c = _read_sync_state()
                wf_c = _auto_build_workflow(reply[:320] or q_send[:320], str(st_c.get("active_model") or "qwen-image-21-turbo"), str(st_c.get("active_lora") or ""), float(st_c.get("lora_strength", 1.0)))
                _write_sync_state(workflow=wf_c)
            return {"ok": True, "reply": reply, "model": mid, "tok_s": round(toks / el, 1), "seconds": round(el, 2), "rag_sources": [x["source"] for x in docs]}
        if act == "clear_chat": _ST["history"].clear(); return {"ok": True}
        if act == "gen_image":
            key = str(d.get("key", ""))
            rep_obj = d.get("reply_to") if isinstance(d.get("reply_to"), dict) else None
            rep_txt = str((rep_obj or {}).get("text", "")).strip()
            prompt = _merge_reply_and_prompt(rep_txt, str(d.get("prompt", "")).strip())
            lora_nm, lora_st = str(d.get("lora_name", "")).strip(), float(d.get("lora_strength", 1.0))
            auto_wf = bool(d.get("auto_wf", True))
            if not prompt: return {"ok": False, "error": "พิมพ์คำบรรยายภาพ หรือกด ↩️ รีพายข้อความในแชทที่ต้องการนำมาวาดภาพก่อนครับ"}
            ok_a, ms = _sync_image_preset(key, _MDIR)
            if not ok_a: return {"ok": False, "error": f"โมเดลภาพ '{key}' ยังไม่ครบ ({', '.join(ms)}) — กด 📥 โหลด ในแท็บ 1 ก่อนครับ"}
            wf = _auto_build_workflow(prompt, key, lora_nm, lora_st, use_llm=bool(d.get("use_llm")), port=_PORT) if auto_wf else {"raw": prompt, "pos": prompt, "neg": "blurry, low quality, deformed, watermark", "width": int(d.get("width", 1024)), "height": int(d.get("height", 1024)), "steps": 8, "cfg": 1.0, "sampler": "euler", "scheduler": "simple", "seed": -1, "style_label": "Manual", "summary": "Manual Settings", "wf_ts": time.time()}
            _write_sync_state(active_model=key, active_lora=lora_nm, lora_strength=lora_st, workflow=wf)
            def _iw():
                try:
                    _ST["img_stat"] = f"⏳ Automation ({wf['summary']}) → กำลังวาดด้วย {key}..."
                    res = _run_quick_comfy(key, wf, lora_name=lora_nm, lora_str=lora_st)
                    _ST["last_img"] = {"uri": _ig.image_data_uri(res["path"]), "name": Path(res["path"]).name, "key": key, "seconds": res["seconds"], "size": f"{res['w']}×{res['h']} • {wf['style_label']}", "prompt": prompt}
                    _ST["img_stat"] = f"✅ วาดเสร็จใน {res['seconds']}s ({wf['summary']})"
                except Exception as e:
                    _ST["img_stat"] = f"❌ {e}"
                finally:
                    _ST["job"] = ""; _ST["live_step"] = ""
            _ST["job"] = f"วาดภาพ {key}"
            threading.Thread(target=_iw, daemon=True).start()
            return {"ok": True, "workflow": wf, "used_prompt": prompt}
        if act == "rag_op":
            rop = str(d.get("op", ""))
            if rop == "upload":
                nm, raw = str(d.get("name", "doc.txt")), base64.b64decode(str(d.get("b64", "")))
                if nm.lower().endswith(".pdf"): GLOBAL_RAG.add_pdf_bytes(raw, filename=nm, chunk_size=500)
                else: GLOBAL_RAG.add_documents([{"source": nm, "text": raw.decode("utf-8", errors="ignore")}], chunk_size=500)
                return {"ok": True, "msg": f"✅ นำเข้า {nm} สำเร็จ ({len(GLOBAL_RAG.chunks)} chunks)"}
            if rop == "add_text":
                txt = str(d.get("text", "")).strip()
                if txt: GLOBAL_RAG.add_documents([{"source": "note", "text": txt}], chunk_size=500)
                return {"ok": True, "msg": f"✅ เพิ่มข้อความแล้ว ({len(GLOBAL_RAG.chunks)} chunks)"}
            if rop == "search": return {"ok": True, "hits": GLOBAL_RAG.search(str(d.get("query", "")), top_k=4)}
            if rop == "save":
                p = GLOBAL_RAG.save()
                return {"ok": True, "msg": f"💾 บันทึกคลัง RAG แล้ว ({len(GLOBAL_RAG.chunks)} chunks)"}
            if rop == "load":
                try:
                    n = GLOBAL_RAG.load()
                    return {"ok": True, "msg": f"📂 โหลดคลัง RAG แล้ว ({n} chunks)"}
                except Exception as _e:
                    return {"ok": False, "msg": f"⚠️ ยังไม่มีไฟล์บันทึก ({_e})"}
            if rop == "clear": GLOBAL_RAG.clear(); return {"ok": True, "msg": "🧹 ล้างคลัง RAG แล้ว"}
        return {"ok": False, "error": act}
    except Exception as e:
        return {"ok": False, "error": str(e)}

if IS_COLAB:
    from google.colab import output as colab_output  # type: ignore[import-not-found]
    colab_output.register_callback("strata.studio", lambda act, payload=None: JSON(_handle_action(act, payload)))

_ip = get_ipython()
if _ip is not None and getattr(_ip, "kernel", None) is not None:
    def _comm_target(comm, _open_msg):
        @comm.on_msg
        def _recv(msg):
            body = (msg.get("content") or {}).get("data") or {}
            comm.send({"id": body.get("id", ""), "result": _handle_action(str(body.get("act", "poll")), body.get("payload") or {})})
    with contextlib.suppress(Exception): _ip.kernel.comm_manager.register_target("strata.studio", _comm_target)

_STUDIO_HTML = r"""
<div id="st-root">
<style>
  html, body { margin: 0 !important; padding: 0 !important; width: 100% !important; background: #0b1120 !important; overflow-x: hidden; }
  #st-root { width: 100% !important; min-height: 860px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0b1120; color: #f8fafc; border: 1px solid #334155; border-radius: 10px; padding: 14px 16px; box-sizing: border-box; }
  #st-root * { box-sizing: border-box; touch-action: manipulation; }
  .s-hdr { background: #0f172a; border: 1px solid #334155; border-radius: 10px; padding: 12px 16px; margin-bottom: 10px; }
  .s-bdgs { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 7px; font-size: 12.5px; }
  .s-bdg { padding: 3px 10px; border-radius: 99px; background: #1e293b; border: 1px solid #334155; }
  .s-btn { min-height: 40px; padding: 7px 13px; border-radius: 8px; border: 1px solid #475569; background: #1e293b; color: #f8fafc; font-size: 12.8px; font-weight: 600; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  .s-btn:disabled { opacity: 0.42; cursor: not-allowed; filter: grayscale(0.35); }
  .s-btn.pri { background: #2563eb; border-color: #3b82f6; }
  .s-btn.suc { background: #059669; border-color: #10b981; }
  .s-btn.wrn { background: #d97706; border-color: #f59e0b; }
  .s-btn.dng { background: #dc2626; border-color: #ef4444; }
  .s-btn.act { background: #7c3aed; border-color: #a78bfa; }
  .s-tabs { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin: 10px 0; }
  .s-row { display: flex; flex-wrap: wrap; gap: 7px; align-items: center; margin-bottom: 9px; }
  .s-inp, .s-sel, .s-ta { width: 100%; min-height: 40px; background: #0f172a; color: #f8fafc; border: 1px solid #475569; border-radius: 8px; padding: 7px 11px; font-size: 13.5px; }
  .s-card { background: #0f172a; border: 1px solid #334155; border-radius: 10px; padding: 11px 13px; margin-bottom: 9px; }
  .s-mlist { display: flex; flex-direction: column; gap: 7px; max-height: 760px; overflow-y: auto; padding-right: 4px; }
  .s-mrow { background: #0f172a; border: 1px solid #334155; border-radius: 9px; padding: 10px 13px; display: flex; flex-direction: column; gap: 7px; }
  @media (min-width: 700px) { .s-mrow { flex-direction: row; justify-content: space-between; align-items: center; } }
  .s-tag { display: inline-block; padding: 1px 6px; border-radius: 6px; font-size: 10.5px; margin-left: 4px; }
  .s-feed { background: #0f172a; border: 1px solid #334155; border-radius: 10px; padding: 12px; min-height: 380px; max-height: 640px; overflow-y: auto; margin-bottom: 9px; display: flex; flex-direction: column; gap: 10px; scroll-behavior: smooth; }
  .s-mu { align-self: flex-end; background: #1e3a8a; border: 1px solid #3b82f6; color: #fff; padding: 9px 13px; border-radius: 11px; max-width: 86%; white-space: pre-wrap; font-size: 13.5px; transition: box-shadow 0.35s, border-color 0.35s; }
  .s-mb { align-self: flex-start; background: #1e293b; border: 1px solid #334155; padding: 10px 13px; border-radius: 11px; max-width: 94%; white-space: pre-wrap; font-size: 13.5px; line-height: 1.55; transition: box-shadow 0.35s, border-color 0.35s; }
  .s-flash { border-color: #e879f9 !important; box-shadow: 0 0 0 3px rgba(217,70,239,0.55) !important; }
  .s-qbox { background: rgba(15,23,42,0.72); border-left: 3px solid #c084fc; border-radius: 6px; padding: 5px 9px; margin-bottom: 6px; font-size: 11.8px; color: #e2e8f0; white-space: normal; cursor: pointer; }
  .s-qbox:hover { background: rgba(59,7,100,0.65); border-left-color: #f0abfc; }
  .s-bact { display: flex; flex-wrap: wrap; gap: 5px; margin-top: 7px; padding-top: 6px; border-top: 1px dashed rgba(148,163,184,0.25); white-space: normal; align-items: center; }
  .s-mbtn { padding: 3px 8px; border-radius: 6px; border: 1px solid #475569; background: rgba(15,23,42,0.82); color: #cbd5e1; font-size: 11.2px; font-weight: 600; cursor: pointer; display: inline-flex; align-items: center; gap: 3px; }
  .s-mbtn:hover { border-color: #a855f7; color: #f8fafc; background: #3b0764; }
  .s-mbtn.pri-m { border-color: #8b5cf6; background: #2e1065; color: #e9d5ff; }
  .s-mbtn.img-m { border-color: #f59e0b; background: #451a03; color: #fde68a; }
  .s-mbtn.dng-m { border-color: #7f1d1d; color: #fca5a5; }
  .s-rbar { background: #1e1b4b; border: 1px solid #8b5cf6; border-left: 4px solid #c084fc; border-radius: 8px; padding: 8px 11px; margin-bottom: 6px; display: flex; flex-direction: column; gap: 6px; font-size: 12.2px; }
  .s-hide { display: none !important; }
</style>
  <div class="s-hdr">
    <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
      <b style="font-size:16px;">🎛️ Strata Studio (<span id="s-plat"></span> • __TOTAL_M__ Models • ปุ่มรีพาย Pro + Auto โหนด 4–7)</b>
      <div style="display:flex;gap:6px;"><button class="s-btn dng" onclick="sAct('cancel')">⏹️ ยกเลิกงาน</button><button class="s-btn" onclick="sPoll(true)">🔄 รีเฟรช</button></div>
    </div>
    <div class="s-bdgs" id="s-bdgs"></div>
    <div style="margin-top:7px;background:#1e293b;height:7px;border-radius:99px;overflow:hidden;"><div id="s-dbar" style="height:100%;background:#22c55e;width:8%;"></div></div>
  </div>
  <div id="s-prog" class="s-card s-hide" style="background:#0c4a6e;border-color:#38bdf8;"></div>
  <div id="s-note" class="s-card s-hide"></div>
  <details class="s-card" open>
    <summary style="cursor:pointer;font-weight:700;color:#e9d5ff;font-size:13px;">⚡ เร่งสปีด Speculative (ค่าเริ่มต้น ✨ Auto = โหลด MODEL Draft คู่กันอัตโนมัติเมื่อกดโหลดโมเดลข้อความ)</summary>
    <div class="s-row" style="margin-top:8px;" id="s-spec-btns"></div>
    <div class="s-row">
      <select id="s-draft-dd" class="s-sel" style="flex:1 1 240px;"></select>
      <button class="s-btn pri" onclick="sAct('dl_draft',{key:document.getElementById('s-draft-dd').value})">📥 โหลด Draft เฉพาะตัว</button>
      <label style="font-size:12.5px;"><input type="checkbox" id="s-kvq8" checked onchange="sSaveSpec(false)"> 💾 KV Q8_0</label>
      <button class="s-btn" onclick="sSaveSpec(true)">🔄 รีสตาร์ทใช้โหมดนี้</button>
    </div>
  </details>
  <div class="s-tabs">
    <button class="s-btn act" id="s-tb-0" onclick="sTab(0)">📦 1. โมเดล & Uncensored LoRA (__TOTAL_M__)</button>
    <button class="s-btn" id="s-tb-1" onclick="sTab(1)">💬 2. แชท + ปุ่มรีพาย Pro</button>
    <button class="s-btn" id="s-tb-2" onclick="sTab(2)">📚 3. คลัง PDF RAG</button>
  </div>
  <div id="s-tp-0">
    <div class="s-row">
      <input id="s-q" class="s-inp" style="flex:1 1 220px;" placeholder="🔍 ค้นหาโมเดล หรือพิมพ์ lora, uncensored, z-image, qwen-image, glm4, stheno..." oninput="sDrawModels()">
      <select id="s-flt" class="s-sel" style="flex:0 1 285px;" onchange="sDrawModels()">
        <option value="all">📋 ทั้งหมด (__TOTAL_M__ รุ่น)</option><option value="installed">✅ ติดตั้งแล้ว</option><option value="fit">🎮 พอดี VRAM จริง</option>
        <option value="lora">🔥🧩 โมเดลเสริม Uncensored LoRA (6)</option><option value="image">🎨 โมเดลสร้างภาพ GGUF (6)</option>
        <option value="obliterated">💥 สาย OBLITERATED</option><option value="unrestricted">🔓 สาย Unrestricted</option><option value="rp">🎭 สาย Roleplay (RP / ERP)</option>
        <option value="heretic">🔥 สาย Heretic / Uncensored</option><option value="code">💻 สาย Code / Security</option>
      </select>
    </div>
    <div class="s-row">
      <button class="s-btn wrn" onclick="sAct('model_op',{op:'stop_server'})">🛑 หยุด Server</button>
      <button class="s-btn" onclick="sAct('model_op',{op:'clean_cache'})">🧹 ล้างแคช</button>
      <button class="s-btn dng" onclick="if(confirm('ยืนยันลบโมเดลทั้งหมด?')) sAct('model_op',{op:'delete_all'})">🗑️ ลบทั้งหมด</button>
    </div>
    <div class="s-mlist" id="s-mlist"></div>
  </div>
  <div id="s-tp-1" class="s-hide">
    <div class="s-row">
      <select id="s-tpl" class="s-sel" style="flex:1 1 220px;" onchange="sUseTpl()"></select>
      <select id="s-qsw" class="s-sel" style="flex:1 1 180px;"></select>
      <button class="s-btn wrn" onclick="const k=document.getElementById('s-qsw').value; if(k) sAct('model_op',{op:'run',key:k})">⚡ สลับ</button>
    </div>
    <textarea id="s-sys" class="s-ta" rows="2" style="margin-bottom:7px;"></textarea>
    <div class="s-row" style="font-size:12.5px;">
      <label><input type="checkbox" id="s-rag" checked> 📚 ใช้ RAG</label>
      <span>Temp: <input id="s-temp" type="number" step="0.05" value="0.7" style="width:60px;padding:3px;background:#1e293b;color:#fff;border:1px solid #475569;border-radius:5px;"></span>
      <span>Min-P: <input id="s-minp" type="number" step="0.01" value="0.05" style="width:60px;padding:3px;background:#1e293b;color:#fff;border:1px solid #475569;border-radius:5px;"></span>
      <span>DRY: <input id="s-dry" type="number" step="0.1" value="0" style="width:56px;padding:3px;background:#1e293b;color:#fff;border:1px solid #475569;border-radius:5px;"></span>
      <span>MaxTok: <input id="s-max" type="number" step="64" value="1024" style="width:72px;padding:3px;background:#1e293b;color:#fff;border:1px solid #475569;border-radius:5px;"></span>
    </div>
    <div class="s-feed" id="s-feed"><div style="color:#94a3b8;text-align:center;padding:14px;">💬 <b>ระบบรีพายข้อความ Pro:</b> ลากคลุมเฉพาะประโยคที่สนใจแล้วกด <b>↩️ รีพาย</b> ได้ทันที! คลิกที่กล่องอ้างอิงเพื่อเลื่อนกลับไปดูข้อความต้นทาง หรือใช้ปุ่มด่วน <b>🎨 วาดภาพนี้ / 🪄 ส่งเข้าโหนด 4–7 / ✏️ แก้ไข / 📋 คัดลอก</b> ได้บนทุกข้อความ (กด <code>Ctrl+Enter</code> ส่งแชท • <code>Esc</code> ยกเลิกรีพาย)</div></div>
    <div id="s-rbar" class="s-rbar s-hide" style="display:none;">
      <div style="display:flex;justify-content:space-between;align-items:flex-start;gap:8px;">
        <div style="flex:1;min-width:0;cursor:pointer;" onclick="if(replyCtx&&replyCtx.id) sJumpMsg(replyCtx.id)" title="คลิกเพื่อเลื่อนไปดูข้อความต้นทาง">
          <div style="color:#d8b4fe;font-weight:700;margin-bottom:2px;">↩️ กำลังรีพาย <span id="s-rwho"></span> <span id="s-rsel-tag" style="color:#fde047;font-size:11px;"></span></div>
          <div id="s-rtxt-prev" style="color:#cbd5e1;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;"></div>
        </div>
        <button type="button" class="s-mbtn dng-m" onclick="sClearReply()">✖ ยกเลิก (Esc)</button>
      </div>
      <div style="display:flex;flex-wrap:wrap;gap:5px;">
        <button type="button" class="s-mbtn pri-m" onclick="sChat(false)">📨 ส่งถามต่อ LLM ทันที</button>
        <button type="button" class="s-mbtn img-m" onclick="sGenImg()">🖼️ สร้างภาพจากที่รีพาย</button>
        <button type="button" class="s-mbtn" onclick="sPushAutoWf(true,false)">🤖 ให้ LLM แต่งโหนด 4–7</button>
        <button type="button" class="s-mbtn" onclick="sEditFromReply()">✏️ ดึงข้อความลงช่องพิมพ์</button>
      </div>
    </div>
    <textarea id="s-uin" class="s-ta" rows="3" oninput="sDebouncedAutoWf()" onkeydown="sOnKey(event)" placeholder="พิมพ์แชท หรือคำบรรยายภาพไทย/อังกฤษ (Ctrl+Enter ส่งแชท • Shift+Ctrl+Enter สร้างภาพ • Esc ยกเลิกรีพาย)..."></textarea>
    <div class="s-row" style="margin-top:6px;">
      <button class="s-btn suc" style="flex:2 1 140px;" id="s-sbtn" onclick="sChat(false)">📨 ส่งข้อความ / ถามต่อ (LLM)</button>
      <button class="s-btn wrn" style="flex:2 1 150px;" id="s-imbtn-top" disabled onclick="sGenImg()">🔒 โหลดโมเดลภาพก่อน</button>
      <button class="s-btn pri" style="flex:1 1 95px;" onclick="sChat(true)">🔁 สุ่มใหม่</button>
      <button class="s-btn dng" onclick="sClrChat()">🗑️ ล้างแชท</button>
    </div>
    <div class="s-card" style="margin-top:8px;border-color:#f59e0b;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;margin-bottom:6px;">
        <span style="font-size:13px;font-weight:700;color:#fbbf24;">🤖🎨 Automation Workflow (ซิงก์ข้อความ & ข้อความที่รีพาย → โหนด 4–7 เซลล์ 3 อัตโนมัติ)</span>
        <label style="font-size:12px;color:#fde68a;"><input type="checkbox" id="s-autowf" checked onchange="sDebouncedAutoWf()"> ⚡ ตั้งค่าโหนด 4–7 อัตโนมัติตาม Prompt</label>
      </div>
      <div class="s-row">
        <select id="s-imk" class="s-sel" style="flex:1 1 175px;" onchange="sOnImgSel()"></select>
        <select id="s-imlora" class="s-sel" style="flex:1 1 175px;" onchange="sOnLoraSel()"></select>
        <span style="font-size:12px;">LoRA×<input id="s-imlstr" type="number" step="0.05" value="1.00" style="width:56px;padding:4px;background:#0f172a;color:#fde047;border:1px solid #f59e0b;border-radius:6px;" onchange="sOnLoraSel()"></span>
        <select id="s-imsz" class="s-sel" style="flex:0 1 150px;"><option value="1024x1024">1024×1024 (1:1)</option><option value="832x1216">832×1216 (แนวตั้ง)</option><option value="1216x832">1216×832 (แนวนอน)</option><option value="768x768">768×768 (เร็ว)</option></select>
      </div>
      <div class="s-row">
        <button class="s-btn act" onclick="sPushAutoWf(false,false)">🪄 ซิงก์โหนด 4–7 เดี๋ยวนี้</button>
        <button class="s-btn pri" onclick="sPushAutoWf(true,false)" title="ใช้ LLM ที่เปิดอยู่ช่วยแต่ง Positive/Negative Prompt ภาษาอังกฤษ">🤖 ให้ LLM แต่งโหนด 4–7</button>
        <button class="s-btn" onclick="sPushAutoWf(false,true)">💬 ดึงจากแชทล่าสุด</button>
        <button class="s-btn wrn" style="flex:1 1 150px;" id="s-imbtn" disabled onclick="sGenImg()">🔒 โหลดโมเดลภาพก่อน</button>
      </div>
      <div id="s-wf-prev" style="font-size:11.8px;color:#a7f3d0;background:#022c22;border:1px solid #065f46;border-radius:7px;padding:6px 9px;margin-bottom:5px;" class="s-hide"></div>
      <div id="s-imst" style="font-size:12px;color:#38bdf8;"></div>
    </div>
  </div>
  <div id="s-tp-2" class="s-hide">
    <div class="s-card">
      <div class="s-row">
        <input type="file" id="s-fup" multiple accept=".pdf,.txt,.md,.csv,.json,.py" class="s-inp" style="flex:1 1 200px;">
        <button class="s-btn suc" onclick="sUpRag()">📥 นำไฟล์เข้า RAG</button>
        <button class="s-btn pri" onclick="sRag('save',{})" title="บันทึกคลังเวกเตอร์ลงไฟล์ rag_store.npz">💾 บันทึกคลัง</button>
        <button class="s-btn" onclick="sRag('load',{})" title="โหลดคลังเวกเตอร์จากไฟล์ rag_store.npz">📂 โหลดคลัง</button>
        <button class="s-btn dng" onclick="sRag('clear',{}) ">🗑️ ล้างคลัง</button>
      </div>
      <textarea id="s-rtxt" class="s-ta" rows="4" placeholder="วางข้อความ/ข้อมูลตัวละครเข้าคลังความรู้..."></textarea>
      <div class="s-row" style="margin-top:6px;">
        <button class="s-btn pri" onclick="sAddTxt()">➕ เพิ่มข้อความ</button>
        <input id="s-rq" class="s-inp" style="flex:1 1 160px;" placeholder="คำค้นทดสอบ...">
        <button class="s-btn" onclick="sSearchRag()">🔍 ทดสอบค้นหา</button>
      </div>
      <div id="s-rout" style="font-size:12.5px;color:#cbd5e1;margin-top:6px;"></div>
    </div>
  </div>
</div>
<script>
(function(){
  const INIT = __INIT_JSON__;
  let S = INIT.state; const TPLS = INIT.templates; let shownImg = ''; let wfTimer = null;
  let _comm = null; const _waiters = {};
  const MSG_MAP = {}; let msgSeq = 0; window.replyCtx = null;
  function syncCellHeight(){ try{ if(typeof google !== 'undefined' && google.colab && google.colab.output && google.colab.output.setIframeHeight){ google.colab.output.setIframeHeight(Math.max(880, document.getElementById('st-root').scrollHeight + 16), true, {maxHeight: 8000}); } }catch(_e){} }
  function esc(x){ const d=document.createElement('div'); d.textContent=String(x||''); return d.innerHTML; }
  function regMsg(who, text){ const id = 'm' + (++msgSeq); MSG_MAP[id] = {id, who, text: String(text||'')}; return id; }

  function getSelectedInsideBubble(mid){
    try{
      const sel = window.getSelection ? window.getSelection() : null;
      if(!sel || sel.isCollapsed) return '';
      const txt = String(sel.toString() || '').trim();
      if(!txt) return '';
      const bub = document.getElementById('b-' + mid);
      if(bub && sel.anchorNode && bub.contains(sel.anchorNode)) return txt;
    }catch(_e){}
    return '';
  }

  function replyButtonsHtml(mid){
    return `<div class="s-bact">`
      + `<button type="button" class="s-mbtn pri-m" title="ลากคลุมข้อความบางส่วนก่อนกดเพื่อรีพายเฉพาะส่วนได้" onclick="sReplyMsg('${mid}','reply')">↩️ รีพาย</button>`
      + `<button type="button" class="s-mbtn img-m" onclick="sReplyMsg('${mid}','draw')">🎨 วาดภาพนี้</button>`
      + `<button type="button" class="s-mbtn" onclick="sReplyMsg('${mid}','syncwf')">🪄 ส่งเข้าโหนด 4–7</button>`
      + `<button type="button" class="s-mbtn" onclick="sEditMsg('${mid}')">✏️ แก้ไข</button>`
      + `<button type="button" class="s-mbtn" id="cp-${mid}" onclick="sCopyMsg('${mid}')">📋 คัดลอก</button>`
      + `<button type="button" class="s-mbtn dng-m" title="ลบกล่องข้อความนี้" onclick="sDelMsg('${mid}')">🗑️</button>`
      + `</div>`;
  }

  function quoteBoxHtml(rep){
    if(!rep || !rep.text) return '';
    const shortTxt = rep.text.length > 140 ? rep.text.slice(0,140) + '...' : rep.text;
    const jmp = rep.id ? `onclick="sJumpMsg('${esc(rep.id)}')" title="คลิกเพื่อเลื่อนไปข้อความต้นทาง"` : '';
    const ptag = rep.partial ? ' <span style="color:#fde047;">(ข้อความที่คลุมเลือก)</span>' : '';
    return `<div class="s-qbox" ${jmp}><b>↩️ ${esc(rep.who||'ข้อความที่รีพาย')}</b>${ptag}: ${esc(shortTxt)}</div>`;
  }

  window.sJumpMsg = function(mid){
    const el = document.getElementById('b-' + mid);
    if(!el) return;
    el.scrollIntoView({behavior: 'smooth', block: 'center'});
    el.classList.add('s-flash');
    setTimeout(()=>{ el.classList.remove('s-flash'); }, 1300);
  };

  window.sCopyMsg = async function(mid){
    const item = MSG_MAP[mid]; if(!item) return;
    try{ await navigator.clipboard.writeText(item.text); }catch(_e){ const ta=document.createElement('textarea'); ta.value=item.text; document.body.appendChild(ta); ta.select(); document.execCommand('copy'); document.body.removeChild(ta); }
    const bt = document.getElementById('cp-' + mid);
    if(bt){ const old = bt.textContent; bt.textContent = '✅ คัดลอกแล้ว'; setTimeout(()=>{ bt.textContent = old; }, 1200); }
  };

  window.sEditMsg = function(mid){
    const item = MSG_MAP[mid]; if(!item) return;
    const selTxt = getSelectedInsideBubble(mid);
    const u = document.getElementById('s-uin');
    u.value = selTxt || item.text;
    u.focus(); sDebouncedAutoWf(); syncCellHeight();
  };

  window.sEditFromReply = function(){
    if(!window.replyCtx) return;
    const u = document.getElementById('s-uin');
    u.value = window.replyCtx.text;
    sClearReply(); u.focus();
  };

  window.sDelMsg = function(mid){
    const el = document.getElementById('b-' + mid);
    if(el) el.remove();
    if(window.replyCtx && window.replyCtx.id === mid) sClearReply();
    syncCellHeight();
  };

  window.sOnKey = function(ev){
    if(ev.key === 'Escape' && window.replyCtx){ ev.preventDefault(); sClearReply(); return; }
    if(ev.key === 'Enter' && (ev.ctrlKey || ev.metaKey)){
      ev.preventDefault();
      if(ev.shiftKey) sGenImg(); else sChat(false);
    }
  };

  window.sReplyMsg = function(mid, mode){
    const item = MSG_MAP[mid]; if(!item) return;
    const selTxt = getSelectedInsideBubble(mid);
    const useTxt = selTxt || item.text;
    window.replyCtx = {id: item.id, who: item.who, text: useTxt, partial: !!selTxt};
    const bar = document.getElementById('s-rbar');
    document.getElementById('s-rwho').textContent = '[' + item.who + ']';
    document.getElementById('s-rsel-tag').textContent = selTxt ? '✂️ รีพายเฉพาะส่วนที่ลากคลุม' : '🔗 คลิกเพื่อย้อนดูต้นทาง';
    document.getElementById('s-rtxt-prev').textContent = useTxt.split('\\n').join(' ').slice(0, 190);
    bar.style.display = 'flex'; bar.classList.remove('s-hide');
    sUpdImgBtn();
    if(mode === 'syncwf'){ sPushAutoWf(false, false); document.getElementById('s-imst').textContent = '🪄 ซิงก์ข้อความที่รีพายเข้าโหนด 4–7 เรียบร้อย!'; syncCellHeight(); return; }
    sDebouncedAutoWf();
    const u = document.getElementById('s-uin');
    if(mode === 'draw'){
      const curM = S.models.find(m=>m.key===document.getElementById('s-imk').value);
      if(curM && curM.installed){ sGenImg(); return; }
      document.getElementById('s-imst').textContent = '🔒 ซิงก์ข้อความเข้าโหนด 4–7 แล้ว — กรุณาโหลดโมเดลภาพในแท็บ 1 ก่อนกดสร้างภาพครับ';
    }
    if(u){ u.focus(); }
    syncCellHeight();
  };

  window.sClearReply = function(){
    window.replyCtx = null;
    const bar = document.getElementById('s-rbar');
    bar.classList.add('s-hide'); bar.style.display = 'none';
    sUpdImgBtn(); sDebouncedAutoWf(); syncCellHeight();
  };

  async function callRpc(a,p){
    if(typeof google !== 'undefined' && google.colab && google.colab.kernel){
      const r = await google.colab.kernel.invokeFunction('strata.studio',[a,p||{}],{});
      return (r && r.data && r.data['application/json']) ? r.data['application/json'] : r;
    }
    if(typeof Jupyter !== 'undefined' && Jupyter.notebook && Jupyter.notebook.kernel){
      if(!_comm){ _comm = Jupyter.notebook.kernel.comm_manager.new_comm('strata.studio', {}); _comm.on_msg(m => { const d=(m.content||{}).data||{}; if(_waiters[d.id]){ _waiters[d.id](d.result); delete _waiters[d.id]; } }); }
      return new Promise(res => { const id = String(Date.now()) + Math.random(); _waiters[id] = res; _comm.send({id, act: a, payload: p||{}}); });
    }
    return null;
  }
  function showWfPreview(wf){
    const bx = document.getElementById('s-wf-prev'); if(!bx || !wf) return;
    bx.classList.remove('s-hide');
    const szEl = document.getElementById('s-imsz'); const szVal = `${wf.width}x${wf.height}`;
    if(szEl && [...szEl.options].some(o=>o.value===szVal)) szEl.value = szVal;
    bx.innerHTML = `<b>🔗 ซิงก์เข้าโหนด 4–7 เซลล์ 3 แล้ว:</b> ${esc(wf.summary)}<br><span style="color:#86efac;">➕ <b>Node 4 (Pos):</b> ${esc(wf.pos.slice(0,135))}${wf.pos.length>135?'...':''}</span><br><span style="color:#fca5a5;">➖ <b>Node 5 (Neg):</b> ${esc(wf.neg.slice(0,115))}${wf.neg.length>115?'...':''}</span>`;
    syncCellHeight();
  }
  window.sPushAutoWf = async function(useLlm, fromChat){
    const u = document.getElementById('s-uin'); const k = document.getElementById('s-imk').value;
    const l = document.getElementById('s-imlora').value; const ls = parseFloat(document.getElementById('s-imlstr').value||'1.0');
    if(useLlm) document.getElementById('s-imst').textContent = '🤖 กำลังให้ LLM วิเคราะห์และแต่งโหนด 4–7...';
    const r = await callRpc('auto_workflow',{prompt:u.value,reply_to:window.replyCtx,key:k,lora_name:l,lora_strength:ls,use_llm:!!useLlm,from_last_chat:!!fromChat});
    if(r && r.ok && r.workflow){
      if(fromChat && r.from_chat && !u.value.trim() && !window.replyCtx) u.value = r.from_chat;
      showWfPreview(r.workflow);
      if(useLlm) document.getElementById('s-imst').textContent = '✅ แต่งและซิงก์โหนด 4–7 ไปยังเซลล์ที่ 3 เรียบร้อย!';
    }
  };
  window.sDebouncedAutoWf = function(){ const chk = document.getElementById('s-autowf'); if(!chk || !chk.checked) return; if(wfTimer) clearTimeout(wfTimer); wfTimer = setTimeout(()=>{ sPushAutoWf(false, false); }, 420); };
  window.sAct = async function(a,p){ const r=await callRpc(a,p); if(r && r.models){ S=r; drawTop(); sDrawModels(); syncCellHeight(); } return r; };
  window.sTab = function(i){ for(let k=0;k<3;k++){ document.getElementById('s-tp-'+k).classList.toggle('s-hide',k!==i); document.getElementById('s-tb-'+k).classList.toggle('act',k===i); } sUpdImgBtn(); setTimeout(syncCellHeight, 30); };
  window.sSaveSpec = function(rst){ sAct('set_spec',{mode:(S.spec&&S.spec.mode)||'auto',draft_max:0,kv_q8:document.getElementById('s-kvq8').checked,restart:!!rst}); };
  window.sPickSpec = function(m){ if(!S.spec) S.spec={}; S.spec.mode=m; sSaveSpec(false); };
  window.sGoImg = async function(k){
    const m = S.models.find(x=>x.key===k); if(!m || !m.installed) return;
    await sAct('link_image',{key:k});
    if(m.kind === 'image'){ const sel=document.getElementById('s-imk'); if(sel) sel.value=k; }\n    if(m.kind === 'lora' && m.lora_file){ const lsel=document.getElementById('s-imlora'); if(lsel) lsel.value=m.lora_file; }
    sUpdImgBtn(); sTab(1); sDebouncedAutoWf();
    const u=document.getElementById('s-uin'); if(u) u.focus();
  };
  window.sOnImgSel = function(){ sUpdImgBtn(); const v=document.getElementById('s-imk').value; const m=S.models.find(x=>x.key===v); if(m && m.installed) callRpc('sync_comfy_sel',{active_model:v,prompt:document.getElementById('s-uin').value}).then(r=>{if(r&&r.sync&&r.sync.workflow) showWfPreview(r.sync.workflow);}); };
  window.sOnLoraSel = function(){ callRpc('sync_comfy_sel',{active_lora:document.getElementById('s-imlora').value,lora_strength:parseFloat(document.getElementById('s-imlstr').value||'1.0'),prompt:document.getElementById('s-uin').value}).then(r=>{if(r&&r.sync&&r.sync.workflow) showWfPreview(r.sync.workflow);}); };
  window.sUpdImgBtn = function(){
    const sel = document.getElementById('s-imk'); const lsel = document.getElementById('s-imlora');
    const btn = document.getElementById('s-imbtn'); const btnTop = document.getElementById('s-imbtn-top');
    const st = document.getElementById('s-imst'); if(!sel || !btn) return;
    const prev = sel.value; const imgModels = (S.models||[]).filter(m=>m.kind==='image');
    sel.innerHTML = imgModels.map(m=>`<option value="${esc(m.key)}">${m.installed?'✅':'⬜ (ยังไม่โหลด)'} ${esc(m.key)}</option>`).join('');
    const syncM = S.sync && S.sync.active_model;
    if(syncM && imgModels.some(m=>m.key===syncM && m.installed)) sel.value = syncM;
    else if(prev && imgModels.some(m=>m.key===prev)) sel.value = prev;
    else { const rdy = imgModels.find(m=>m.installed); if(rdy) sel.value = rdy.key; }
    if(lsel){
      const prevL = lsel.value; const rl = S.ready_loras || [];
      lsel.innerHTML = '<option value="">🧩 ไม่ใช้ Uncensored LoRA เพิ่ม</option>' + rl.map(x=>`<option value="${esc(x.file)}">${esc(x.label)}</option>`).join('');
      const syncL = S.sync && S.sync.active_lora;
      if(syncL && rl.some(x=>x.file===syncL)) lsel.value = syncL;
      else if(prevL && rl.some(x=>x.file===prevL)) lsel.value = prevL;
    }
    const cur = imgModels.find(m=>m.key===sel.value);
    const lblOk = window.replyCtx ? '🖼️ สร้างภาพจากที่รีพาย' : '🖼️ สร้างภาพ (Auto โหนด 4–7)';
    if(!cur || !cur.installed){ btn.disabled = true; btn.textContent = '🔒 โหลดโมเดลภาพก่อน'; if(btnTop){ btnTop.disabled = true; btnTop.textContent = '🔒 โหลดโมเดลภาพก่อน'; } }
    else if(S.job && S.job.startsWith('วาดภาพ')){ btn.disabled = true; btn.textContent = '⏳ กำลังวาดภาพ...'; if(btnTop){ btnTop.disabled = true; btnTop.textContent = '⏳ กำลังวาดภาพ...'; } }
    else { btn.disabled = false; btn.textContent = lblOk; if(btnTop){ btnTop.disabled = false; btnTop.textContent = lblOk; } if(!S.img_stat && st.textContent.startsWith('🔒')) st.textContent = ''; }
  };
  function drawTop(){
    const specMode = (S.spec && S.spec.mode) || 'auto';
    document.getElementById('s-plat').textContent = S.platform + ' • Port ' + S.port;
    document.getElementById('s-kvq8').checked = !!(S.spec && S.spec.kv_q8);
    const sb = S.online ? `<span class="s-bdg" style="background:#14532d;color:#4ade80;">🟢 ${esc(S.active_model)}</span>` : `<span class="s-bdg" style="background:#450a0a;color:#fca5a5;">🔴 Offline (:${S.port})</span>`;
    const syncImgTxt = (S.sync && S.sync.active_model) ? `<span class="s-bdg" style="background:#064e3b;color:#6ee7b7;">🎨 ซิงก์เซลล์ 3: ${esc(S.sync.active_model)}</span>` : '';
    document.getElementById('s-bdgs').innerHTML = `${sb} ${syncImgTxt} <span class="s-bdg">📚 RAG: ${S.rag_chunks}</span> <span class="s-bdg">${esc(S.gpu)}</span> <span class="s-bdg">💾 ว่าง ${S.disk.free_gb}/${S.disk.total_gb} GB</span> <span class="s-bdg">${esc(S.net)}</span>`;
    document.getElementById('s-dbar').style.width = S.disk.pct + '%';
    const modes = [['auto','✨ Auto (โหลด Draft คู่กันอัตโนมัติ)'],['mtp','🚀 MTP'],['draft','⚡ Draft'],['ngram','📚 N-gram'],['off','⏹️ ปิด']];
    document.getElementById('s-spec-btns').innerHTML = modes.map(([k,l])=>`<button class="s-btn ${specMode===k?'act':''}" onclick="sPickSpec('${k}')">${l}</button>`).join('');
    const dd = document.getElementById('s-draft-dd'); const pv = (S.spec && S.spec.selected_draft_key) || dd.value;
    dd.innerHTML = (S.drafts||[]).map(d=>`<option value="${esc(d.key)}">${esc(d.label)}</option>`).join('');
    if(pv && [...dd.options].some(o=>o.value===pv)) dd.value=pv;
    const pb = document.getElementById('s-prog');
    if(S.progress){
      pb.classList.remove('s-hide');
      pb.innerHTML = `<div style="display:flex;justify-content:space-between;flex-wrap:wrap;font-size:12.5px;margin-bottom:5px;"><span>📥 <b>Fetching [${esc(S.progress.files)}]</b> • ${esc(S.progress.file)}</span><span><b>${S.progress.pct.toFixed(1)}%</b> (${esc(S.progress.done)}/${esc(S.progress.total)}) • <b style="color:#fde047;">⬇ ${esc(S.progress.rate)}</b> • ETA ${esc(S.progress.eta)}</span></div><div style="background:#082f49;height:10px;border-radius:99px;overflow:hidden;"><div style="width:${Math.max(2,S.progress.pct)}%;height:100%;background:linear-gradient(90deg,#38bdf8,#34d399);"></div></div>`;
    } else if(S.job){
      pb.classList.remove('s-hide');
      const detail = S.live_step ? ` — <span style="color:#fde047;">${esc(S.live_step)}</span>` : '';
      pb.innerHTML = `<div style="font-size:12.5px;">⏳ กำลังทำงาน: <b>${esc(S.job)}</b>${detail}</div>`;
    } else pb.classList.add('s-hide');
    const nb = document.getElementById('s-note');
    if(S.notice){ nb.classList.remove('s-hide'); nb.style.background = S.kind==='ok'?'#064e3b':(S.kind==='err'?'#450a0a':(S.kind==='warn'?'#422006':'#0f172a')); nb.textContent = S.notice; } else nb.classList.add('s-hide');
    if(S.img_stat) document.getElementById('s-imst').textContent = S.img_stat;
    sUpdImgBtn();
    if(S.last_img && S.last_img.name !== shownImg){
      shownImg = S.last_img.name;
      const imgMid = regMsg('🖼️ ' + S.last_img.key, S.last_img.prompt || S.last_img.name);
      const fd = document.getElementById('s-feed'); const dv = document.createElement('div'); dv.className='s-mb'; dv.id = 'b-' + imgMid;
      dv.innerHTML = `<div style="font-size:11.5px;color:#fbbf24;margin-bottom:4px;">🖼️ ${esc(S.last_img.key)} • ${esc(S.last_img.size)} • ${S.last_img.seconds}s</div><img src="${S.last_img.uri}" style="max-width:100%;border-radius:8px;display:block;margin-bottom:6px;"/><a href="${S.last_img.uri}" download="${esc(S.last_img.name)}" class="s-btn pri" style="text-decoration:none;font-size:12px;">💾 บันทึกรูป (${esc(S.last_img.name)})</a>${replyButtonsHtml(imgMid)}`;
      fd.appendChild(dv); fd.scrollTop = fd.scrollHeight;
    }
  }
  function renderFitBadge(m){
    const lb = esc(m.fit_label || '');
    const cols = {ok:'#052e16|#4ade80', moe:'#1e1b4b|#a5b4fc', warn:'#422006|#fbbf24', over:'#450a0a|#fca5a5'};
    const [bg,fg] = (cols[m.fit]||'#1e293b|#94a3b8').split('|');
    return `<span class="s-tag" style="background:${bg};color:${fg};">${lb || '⏳ กำลังวัดจริง...'}</span>`;
  }
  window.sDrawModels = function(){
    const q = (document.getElementById('s-q').value||'').toLowerCase().trim();
    const f = document.getElementById('s-flt').value;
    const specMode = (S.spec && S.spec.mode) || 'auto';
    const items = (S.models || []).filter(m => {
      m.tags = Array.isArray(m.tags) ? m.tags : [];
      if(q && !(m.key.toLowerCase().includes(q)||m.desc.toLowerCase().includes(q)||m.repo.toLowerCase().includes(q))) return false;
      if(f==='installed') return m.installed;
      if(f==='fit') return m.fit==='ok'||m.fit==='moe';
      if(f==='lora') return m.kind==='lora';
      if(f==='image') return m.kind==='image';
      if(f==='code') return m.tags.includes('code')||m.tags.includes('security');
      if(f!=='all') return m.tags.includes(f);
      return true;
    });
    document.getElementById('s-mlist').innerHTML = items.map(m => {
      const lr = m.kind==='lora'?`<span class="s-tag" style="background:#7f1d1d;color:#fde047;font-weight:700;">🔥🧩 UNCENSORED LORA</span>`:'' ;
      const im = m.kind==='image'?`<span class="s-tag" style="background:#0c4a6e;color:#7dd3fc;">🎨 IMAGE</span>`:'';
      const ob = m.tags.includes('obliterated')?`<span class="s-tag" style="background:#7f1d1d;color:#fecaca;">💥 OBLITERATED</span>`:'';
      const rp = m.tags.includes('rp')?`<span class="s-tag" style="background:#500724;color:#f9a8d4;">🎭 RP</span>`:'';
      const hr = (m.tags.includes('heretic')&&m.kind==='text'&&!m.tags.includes('obliterated'))?`<span class="s-tag" style="background:#450a0a;color:#fca5a5;">🔥 HERETIC</span>`:'';
      const ad = (m.kind==='text' && m.auto_draft && specMode==='auto')?`<span class="s-tag" style="background:#2e1065;color:#ddd6fe;">✨ Auto Draft: ${esc(m.auto_draft)}</span>`:'';
      const st = m.installed?`<span style="color:#4ade80;font-size:11.5px;">✅ บนดิสก์ (${esc(m.size_txt)})</span>`:`<span style="color:#94a3b8;font-size:11.5px;">📦 ไฟล์จริง: ${esc(m.size_txt)}</span>`;
      const runB = m.kind==='image' ? `<button class="s-btn suc" ${m.installed?'':'disabled'} onclick="sGoImg('${m.key}')">${m.installed?'🎨 ไปสร้างภาพ':'🔒 ไปสร้างภาพ'}</button>`
                 : (m.kind==='lora' ? `<button class="s-btn wrn" ${m.installed?'':'disabled'} onclick="sGoImg('${m.key}')">${m.installed?'🔥 ใช้ LoRA นี้':'🔒 ใช้ LoRA นี้'}</button>`
                 : `<button class="s-btn suc" ${m.installed?'':'disabled'} onclick="sRunM('${m.key}')">🚀 รัน & แชท</button>`);
      const dlLbl = (m.kind==='text' && specMode==='auto' && m.auto_draft && !m.installed) ? 'โหลด + Draft' : (m.installed ? 'โหลดซ้ำ' : 'โหลด');
      return `<div class="s-mrow"><div><div><b style="color:#38bdf8;">${esc(m.key)}</b>${lr}${im}${ob}${rp}${hr}${ad} ${st} ${renderFitBadge(m)}</div><div style="font-size:11.5px;color:#cbd5e1;margin-top:2px;">${esc(m.desc)} <span style="color:#64748b;">(${esc(m.repo)})</span></div></div><div style="display:flex;gap:5px;flex-wrap:wrap;"><button class="s-btn ${m.installed?'':'pri'}" onclick="sAct('model_op',{op:'download',key:'${m.key}'})">📥 ${dlLbl}</button>${runB}${m.can_del?`<button class="s-btn dng" onclick="sAct('model_op',{op:'delete',key:'${m.key}'})">🗑️ ลบ</button>`:''}</div></div>`;
    }).join('') || '<div style="padding:12px;color:#94a3b8;text-align:center;">ไม่พบโมเดลตามตัวกรอง</div>';
    sUpdImgBtn(); setTimeout(syncCellHeight, 20);
  };
  window.sRunM = async function(k){ await sAct('model_op',{op:'run',key:k}); sTab(1); };
  window.sUseTpl = function(){ const p=TPLS[document.getElementById('s-tpl').value]; if(!p)return; document.getElementById('s-sys').value=p.prompt; document.getElementById('s-temp').value=p.temp; document.getElementById('s-minp').value=p.min_p; document.getElementById('s-dry').value=p.dry; document.getElementById('s-max').value=p.max_tokens; };
  window.sChat = async function(regen){
    const u=document.getElementById('s-uin'); const p=u.value.trim(); const fd=document.getElementById('s-feed');
    const sentReply = window.replyCtx ? Object.assign({}, window.replyCtx) : null;
    if(!regen){
      if(!p && !sentReply) return;
      u.value=''; sClearReply();
      const shownUserTxt = p || '(ขยายความข้อความที่รีพาย)';
      const umid = regMsg('👤 คุณ', sentReply ? `${sentReply.text} ${p}`.trim() : shownUserTxt);
      const ud=document.createElement('div'); ud.className='s-mu'; ud.id = 'b-' + umid;
      ud.innerHTML = `${quoteBoxHtml(sentReply)}<div>👤 ${esc(shownUserTxt)}</div>${replyButtonsHtml(umid)}`;
      fd.appendChild(ud);
    } else if(fd.lastElementChild && fd.lastElementChild.classList.contains('s-mb')) fd.removeChild(fd.lastElementChild);
    const bd=document.createElement('div'); bd.className='s-mb'; bd.textContent='⏳ กำลังคิด...'; fd.appendChild(bd); fd.scrollTop=fd.scrollHeight;
    document.getElementById('s-sbtn').disabled=true;
    try{
      const r=await callRpc('chat',{prompt:p,reply_to:sentReply,regen:!!regen,auto_wf:document.getElementById('s-autowf').checked,system:document.getElementById('s-sys').value,use_rag:document.getElementById('s-rag').checked,temp:parseFloat(document.getElementById('s-temp').value||'0.7'),min_p:parseFloat(document.getElementById('s-minp').value||'0.05'),dry:parseFloat(document.getElementById('s-dry').value||'0'),max_tokens:parseInt(document.getElementById('s-max').value||'1024',10)});
      if(!r||!r.ok){ bd.style.background='#450a0a'; bd.textContent='❌ '+(r&&r.error?r.error:'ผิดพลาด'); }
      else {
        const bmid = regMsg('🤖 ' + r.model, r.reply);
        bd.id = 'b-' + bmid;
        bd.innerHTML=`<div style="font-size:11px;color:#38bdf8;margin-bottom:4px;">🤖 ${esc(r.model)} • <b>${r.tok_s} tok/s</b> (${r.seconds}s)</div><div>${esc(r.reply)}</div>${replyButtonsHtml(bmid)}`;
      }
    } finally { document.getElementById('s-sbtn').disabled=false; fd.scrollTop=fd.scrollHeight; syncCellHeight(); }
  };
  window.sClrChat = async function(){ sClearReply(); await callRpc('clear_chat',{}); document.getElementById('s-feed').innerHTML='<div style="color:#94a3b8;text-align:center;">🧹 ล้างแชทแล้ว</div>'; };
  window.sGenImg = async function(){
    const key = document.getElementById('s-imk').value;
    const cur = S.models.find(m=>m.key===key); if(!cur || !cur.installed){ sUpdImgBtn(); return; }
    const u=document.getElementById('s-uin'); const p=u.value.trim();
    const sentReply = window.replyCtx ? Object.assign({}, window.replyCtx) : null;
    if(!p && !sentReply){ document.getElementById('s-imst').textContent='⚠️ พิมพ์คำบรรยายภาพ หรือกดปุ่ม ↩️ รีพาย ที่ข้อความในแชทก่อนครับ'; return; }
    const [w,h]=document.getElementById('s-imsz').value.split('x');
    const lname = document.getElementById('s-imlora').value || '__AUTO__';
    const lstr = parseFloat(document.getElementById('s-imlstr').value||'1.0');
    const autoWf = document.getElementById('s-autowf').checked;
    const fd=document.getElementById('s-feed');
    const combinedTxt = sentReply ? (p ? `${sentReply.text}, ${p}` : sentReply.text) : p;
    const umid = regMsg('🎨 คำสั่งภาพ', combinedTxt);
    const ud=document.createElement('div'); ud.className='s-mu'; ud.id = 'b-' + umid;
    ud.innerHTML = `${quoteBoxHtml(sentReply)}<div>🎨 ${esc(p || '(สร้างภาพจากข้อความที่รีพาย)')}</div>${replyButtonsHtml(umid)}`;
    fd.appendChild(ud);
    document.getElementById('s-imst').textContent='⏳ กำลังตั้งค่าโหนด 4–7 อัตโนมัติและส่งวาดภาพไปยัง ComfyUI...';
    const r = await callRpc('gen_image',{key,prompt:p,reply_to:sentReply,width:parseInt(w,10),height:parseInt(h,10),lora_name:lname,lora_strength:lstr,auto_wf:autoWf});
    if(r && !r.ok){ document.getElementById('s-imst').textContent='❌ '+r.error; return; }
    if(r && r.workflow) showWfPreview(r.workflow);
    u.value=''; sClearReply(); sPoll(true);
  };
  window.sUpRag = async function(){
    const inp=document.getElementById('s-fup'); if(!inp.files.length)return;
    for(const f of inp.files){
      const b=new Uint8Array(await f.arrayBuffer()); let s=''; for(let i=0;i<b.length;i+=8192) s+=String.fromCharCode.apply(null,b.subarray(i,i+8192));
      const r=await callRpc('rag_op',{op:'upload',name:f.name,b64:btoa(s)});
      document.getElementById('s-rout').textContent=(r&&r.msg)||'';
    }
    sPoll(true);
  };
  window.sAddTxt = async function(){ const el=document.getElementById('s-rtxt'); const r=await callRpc('rag_op',{op:'add_text',text:el.value}); el.value=''; document.getElementById('s-rout').textContent=(r&&r.msg)||''; sPoll(true); };
  window.sSearchRag = async function(){ const r=await callRpc('rag_op',{op:'search',query:document.getElementById('s-rq').value}); document.getElementById('s-rout').innerHTML=((r&&r.hits)||[]).map(h=>`<div class="s-card"><b>#${h.rank} ${esc(h.source)}</b> (${h.score})<br>${esc(h.text)}</div>`).join('')||'ไม่พบข้อมูล'; syncCellHeight(); };
  window.sRag = async function(op,ex){ const r=await callRpc('rag_op',Object.assign({op},ex)); document.getElementById('s-rout').textContent=(r&&r.msg)||''; sPoll(true); };
  let busy=false;
  function sigOf(stObj){ return ((stObj.spec&&stObj.spec.mode)||'auto') + '|' + (stObj.models||[]).map(x=>`${x.installed?1:0}:${x.fit}:${x.fit_label}:${x.size_txt}`).join('|'); }
  window.sPoll = async function(force){
    if(busy&&!force)return; busy=true;
    try{ const r=await callRpc('poll',{}); if(r&&r.models){ const oldSig=sigOf(S); S=r; drawTop(); if(force || sigOf(r)!==oldSig) sDrawModels(); syncCellHeight(); } }catch(_e){} finally{ busy=false; }
  };
  document.getElementById('s-tpl').innerHTML = Object.entries(TPLS).map(([k,v])=>`<option value="${esc(k)}">${esc(v.label)}</option>`).join('');
  document.getElementById('s-qsw').innerHTML = `<option value="">— สลับโมเดลด่วน —</option>` + (S.models||[]).filter(m=>m.kind==='text').map(m=>`<option value="${esc(m.key)}">${esc(m.key)}</option>`).join('');
  sUseTpl(); drawTop(); sDrawModels();
  if(S.sync && S.sync.workflow) showWfPreview(S.sync.workflow);
  if(typeof ResizeObserver !== 'undefined'){ new ResizeObserver(syncCellHeight).observe(document.getElementById('st-root')); }
  syncCellHeight(); setTimeout(syncCellHeight, 250);
  setInterval(()=>{ if(S.job||S.progress||(S.models||[]).some(x=>x.fit==='probing')||(S.img_stat&&S.img_stat.startsWith('⏳'))) sPoll(false); }, 950);
  setInterval(()=>sPoll(false), 4500);
})();
</script>
"""
if IS_KAGGLE:
    from strata.notebook import ensure_widgets
    from strata.ui import show_model_manager_ui
    with contextlib.suppress(Exception): ensure_widgets(platform="kaggle")
    show_model_manager_ui(model_dir=_MDIR, port=_PORT)
else:
    _init_payload = json.dumps({"state": _snap(), "templates": _TPLS}, ensure_ascii=False).replace("</", "<\\\\/")
    display(HTML(_STUDIO_HTML.replace("__INIT_JSON__", _init_payload).replace("__TOTAL_M__", str(len(_cfg.ALL_PRESET_KEYS)))))


In [3]:
# @title 3) 🎨 ComfyUI GGUF Node Workflow Studio (Automation ซิงก์โหนด 4–5 & โหนด 6–7 จากช่องแชทเซลล์ 2 อัตโนมัติ) { display-mode: "form" }
FORCE_RESTART = False  # @param {type:"boolean"}

import contextlib, json, os, random, re, shutil, subprocess, sys, threading, time, urllib.request
from pathlib import Path
from IPython import get_ipython
from IPython.display import HTML, JSON, display

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
SYNC_STATE_FILE = Path("/tmp/_strata_comfy_sync.json")
IS_KAGGLE = any(k.startswith("KAGGLE_") for k in os.environ) or Path("/kaggle/working").is_dir()
IS_COLAB = ("COLAB_RELEASE_TAG" in os.environ) or ("COLAB_GPU" in os.environ and not IS_KAGGLE)

_bases = [Path("/kaggle/working/Colab"), Path("/content/Colab")]
_cands = [Path.cwd(), *Path.cwd().parents, *_bases]
for _b in _bases: _cands += [*sorted(_b.glob("*/")), *sorted(_b.glob("*/*/"))]  # รองรับ repo ที่ห่อโค้ดไว้ในโฟลเดอร์ย่อย เช่น multiple-projects/
for _base in _cands:
    if (_base / "strata" / "__init__.py").is_file():
        while str(_base) in sys.path: sys.path.remove(str(_base))
        sys.path.insert(0, str(_base)); os.chdir(_base); break

_clean_env = {k: v for k, v in os.environ.items() if k != "PYTHONHOME"}
_clean_env["LD_LIBRARY_PATH"] = ":".join(p for p in _clean_env.get("LD_LIBRARY_PATH", "").split(":") if p and not p.startswith("/nix/store/"))
_has_gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0

from strata import config as _cfg, imagegen as _ig
_cfg.COMFY_MODEL_SUBDIRS = ("diffusion_models", "text_encoders", "vae", "loras")
_ig.COMFY_MODEL_SUBDIRS = ("diffusion_models", "text_encoders", "vae", "loras")
COMFY_ROOT = Path(_cfg.COMFY_DIR)
_ig.ensure_comfyui(pull=False)

_PRESETS = {
    "z-image-turbo-heretic": {"targets": {"z-image-turbo-Q4_K_M.gguf": "diffusion_models", "DreamFast-qwen3-4b-heretic.i1-Q4_K_M.gguf": "text_encoders", "split_files/vae/ae.safetensors": "vae"}, "clip_type": "lumina2", "steps": 9, "cfg": 1.0, "sampler": "euler", "scheduler": "simple", "lora_strength": 1.0},
    "qwen-image-21-heretic": {"targets": {"qwen-image-2.1-turbo-Q4_K_M.gguf": "diffusion_models", "comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "clip_type": "qwen_image", "steps": 8, "cfg": 1.0, "sampler": "euler", "scheduler": "simple", "lora_strength": 1.0},
    "qwen-image-21-heretic-q6": {"targets": {"qwen-image-2.1-UC-Q6_K.gguf": "diffusion_models", "comfyui/qwen3-vl-8b-heretic-1.3.0_fp8_e4m3fn.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "clip_type": "qwen_image", "steps": 18, "cfg": 3.0, "sampler": "euler", "scheduler": "simple", "lora_strength": 1.0},
    "chroma1-hd-uncensored": {"targets": {"Chroma1-HD-Q4_K_M.gguf": "diffusion_models", "t5xxl_fp8_e4m3fn_scaled.safetensors": "text_encoders", "split_files/vae/ae.safetensors": "vae"}, "clip_type": "chroma", "steps": 22, "cfg": 4.0, "sampler": "euler", "scheduler": "beta", "lora_strength": 1.0},
    "qwen-image-21-turbo": {"targets": {"qwen-image-2.1-turbo-Q4_K_M.gguf": "diffusion_models", "text_encoders/qwen3vl_8b_int8_convrot.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "clip_type": "qwen_image", "steps": 8, "cfg": 1.0, "sampler": "euler", "scheduler": "simple", "lora_strength": 1.0},
    "qwen-image-21-uncensored": {"targets": {"qwen-image-2.1-UC-Q4_K_M.gguf": "diffusion_models", "text_encoders/qwen3vl_8b_int8_convrot.safetensors": "text_encoders", "vae/qwen_image_2.1_vae_bf16.safetensors": "vae"}, "clip_type": "qwen_image", "steps": 18, "cfg": 3.0, "sampler": "euler", "scheduler": "simple", "lora_strength": 1.0},
}
_LORA_MAP = {
    "lora-qwen21-uncensored": "qwen-image-2.1-uncensored-lora.safetensors",
    "lora-qwen21-omni-uncensored": "chatrlabs-omni-lora-v0.1-bf16.safetensors",
    "lora-zimage-turbo-uncensored": "zimage-turbo-uncensored-lora.safetensors",
    "lora-flux-uncensored-v2": "flux-uncensored-v2.safetensors",
    "lora-flux-uncensored-sagalink": "sagalink-flux-uncensored.safetensors",
    "lora-flux-uncensored-lite": "flux-lora-uncensored-lite.safetensors",
}

def _find_in_sub(sub: Path, rel: str, role: str = "") -> Path | None:
    if not sub.is_dir(): return None
    for c in (sub / rel, sub / Path(rel).name):
        if c.is_file() and not c.name.endswith(".part") and c.stat().st_size > 0: return c.resolve()
    bn = Path(rel).name.lower()
    hit = next((p.resolve() for p in sub.rglob("*") if p.is_file() and not p.name.endswith(".part") and p.name.lower() == bn and p.stat().st_size > 0), None)
    if hit is None and role == "diffusion_models":
        gg = [p.resolve() for p in sub.rglob("*.gguf") if p.is_file() and not p.name.endswith(".part") and p.stat().st_size > 100 * 1024**2]
        if gg: return gg[0]
    return hit

def _sync_from_cell2() -> dict:
    mdir, ldir = Path(_cfg.MODEL_DIR), COMFY_ROOT / "models" / "loras"
    ldir.mkdir(parents=True, exist_ok=True)
    for lk, lfname in _LORA_MAP.items():
        lsub = mdir / lk
        if lsub.is_dir():
            sfs = sorted(p for p in lsub.rglob("*.safetensors") if p.is_file() and not p.name.endswith(".part") and p.stat().st_size > 0)
            if sfs:
                dst = ldir / lfname
                with contextlib.suppress(OSError):
                    if not dst.exists():
                        if dst.is_symlink(): dst.unlink()
                        os.symlink(sfs[0].resolve(), dst)
    ready_map, newest_key, newest_mt = {}, "", -1.0
    for k, sp in _PRESETS.items():
        sub, miss, names, latest_f = mdir / k, [], {}, 0.0
        for rel, role in sp["targets"].items():
            f = _find_in_sub(sub, rel, role)
            actual_name = f.name if f is not None else Path(rel).name
            names[role] = actual_name
            if f is None: miss.append(Path(rel).name)
            else:
                latest_f = max(latest_f, f.stat().st_mtime)
                for r_name in [role, *(["clip"] if role == "text_encoders" else (["unet"] if role == "diffusion_models" else []))]:
                    dst = COMFY_ROOT / "models" / r_name / actual_name
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    with contextlib.suppress(OSError):
                        if dst.is_symlink() or dst.exists(): dst.unlink()
                        os.symlink(f, dst)
        is_rdy = len(miss) == 0
        ready_map[k] = {"key": k, "ready": is_rdy, "missing": miss, "unet": names.get("diffusion_models", ""), "clip": names.get("text_encoders", ""), "vae": names.get("vae", ""), "lora": names.get("loras", ""), "lora_strength": float(sp.get("lora_strength", 1.0)), "clip_type": sp["clip_type"], "steps": sp["steps"], "cfg": sp["cfg"], "sampler": sp["sampler"], "scheduler": sp["scheduler"]}
        if is_rdy and latest_f > newest_mt: newest_mt, newest_key = latest_f, k
    sync_st = {}
    if SYNC_STATE_FILE.is_file():
        with contextlib.suppress(Exception): sync_st = json.loads(SYNC_STATE_FILE.read_text(encoding="utf-8"))
    req_m = str(sync_st.get("active_model") or "")
    active_key = req_m if (req_m in ready_map and ready_map[req_m]["ready"]) else newest_key
    active_spec = ready_map.get(active_key) or {"key": "", "ready": False, "unet": "(รอซิงก์จากเซลล์ 2)", "clip": "(รอซิงก์จากเซลล์ 2)", "vae": "(รอซิงก์จากเซลล์ 2)", "lora": "", "lora_strength": 1.0, "clip_type": "-", "steps": 8, "cfg": 1.0, "sampler": "euler", "scheduler": "simple"}
    avail_loras = sorted(p.name for p in ldir.iterdir() if (p.is_file() or p.is_symlink()) and p.exists() and p.suffix.lower() in (".safetensors", ".pt", ".gguf"))
    req_lora = str(sync_st.get("active_lora") or "")
    chosen_lora = req_lora if (req_lora and req_lora in avail_loras) else (active_spec["lora"] if active_spec["lora"] in avail_loras else "")
    return {"active": active_spec, "loras": avail_loras, "synced_lora": chosen_lora, "synced_lora_strength": float(sync_st.get("lora_strength", active_spec.get("lora_strength", 1.0))), "workflow": sync_st.get("workflow"), "sync_ts": float(sync_st.get("ts", newest_mt if newest_mt > 0 else 0.0))}

def _comfy_up() -> bool:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2.5) as r: return 200 <= int(r.status) < 400
    except Exception: return False

if FORCE_RESTART or not _comfy_up():
    if shutil.which("fuser"): subprocess.run(["fuser", "-k", "8188/tcp"], capture_output=True, check=False)
    _cuda_ok = _has_gpu and subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available()"], env=_clean_env, capture_output=True).returncode == 0
    _cmd = [sys.executable, str(COMFY_ROOT / "main.py"), "--listen", "0.0.0.0", "--port", "8188", "--enable-cors-header", "*", "--preview-method", "none", "--lowvram" if _cuda_ok else "--cpu"]
    _p = subprocess.Popen(_cmd, cwd=str(COMFY_ROOT), stdout=(COMFY_ROOT / "comfyui.log").open("w"), stderr=subprocess.STDOUT, env=_clean_env)
    for _ in range(120):
        if _comfy_up(): break
        if _p.poll() is not None: raise RuntimeError((COMFY_ROOT / "comfyui.log").read_text(errors="ignore")[-1200:])
        time.sleep(1.2)

_CF_STATE = {"running": False, "status": "🟢 ComfyUI :8188 พร้อมทำงาน — พิมพ์หรือรีพายในช่องแชทเซลล์ที่ 2 เพื่อตั้งค่าโหนด 4–7 อัตโนมัติ", "last_img": None}

def _run_comfy_graph(d: dict) -> None:
    import requests
    try:
        with contextlib.suppress(Exception):
            urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/interrupt", data=b"{}", headers={"Content-Type": "application/json"}), timeout=1.5)
            from strata.server import _check_server_online, gpu_devices, release_gpu_memory, stop_strata_server
            if len(gpu_devices(ttl=0.0)) <= 1 and _check_server_online(int(_cfg.DEFAULT_PORT), ttl=0.0)[0]:
                stop_strata_server(int(_cfg.DEFAULT_PORT)); release_gpu_memory(int(_cfg.DEFAULT_PORT)); time.sleep(1.0)
        synced = _sync_from_cell2(); p = synced["active"]
        if not p["ready"] or not p["key"]: _CF_STATE["status"] = "🔒 กรุณากด 📥 โหลด โมเดลภาพในเซลล์ที่ 2 ก่อนครับ"; return
        key = p["key"]
        lora_name = str(d.get("lora_name", synced["synced_lora"])).strip()
        lora_str = round(float(d.get("lora_strength", synced["synced_lora_strength"])), 3)
        use_lora = bool(lora_name) and abs(lora_str) > 1e-6 and (COMFY_ROOT / "models" / "loras" / lora_name).is_file()
        seed = int(d.get("seed", -1))
        if seed < 0: seed = random.randint(1, 2**31 - 1)
        w, h = int(d.get("width", 1024)), int(d.get("height", 1024))
        steps, cfg_v = int(d.get("steps", p["steps"])), float(d.get("cfg", p["cfg"]))
        clip_cls = "CLIPLoaderGGUF" if p["clip"].lower().endswith(".gguf") else "CLIPLoader"
        clip_in = {"clip_name": p["clip"], "type": p["clip_type"]} if clip_cls == "CLIPLoaderGGUF" else {"clip_name": p["clip"], "type": p["clip_type"], "device": "default" if _has_gpu else "cpu"}
        graph = {
            "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": p["unet"]}},
            "2": {"class_type": clip_cls, "inputs": clip_in},
            "3": {"class_type": "CLIPTextEncode", "inputs": {"text": str(d.get("pos", "")), "clip": ["2", 0]}},
            "4": {"class_type": "CLIPTextEncode", "inputs": {"text": str(d.get("neg", "")), "clip": ["2", 0]}},
            "5": {"class_type": "EmptySD3LatentImage", "inputs": {"width": w, "height": h, "batch_size": 1}},
            "6": {"class_type": "KSampler", "inputs": {"model": ["10" if use_lora else "1", 0], "positive": ["3", 0], "negative": ["4", 0], "latent_image": ["5", 0], "seed": seed, "steps": steps, "cfg": cfg_v, "sampler_name": str(d.get("sampler", p["sampler"])), "scheduler": str(d.get("scheduler", p["scheduler"])), "denoise": 1.0}},
            "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["8", 0]}},
            "8": {"class_type": "VAELoader", "inputs": {"vae_name": p["vae"]}},
            "9": {"class_type": "SaveImage", "inputs": {"images": ["7", 0], "filename_prefix": "comfy_gguf"}},
        }
        if use_lora: graph["10"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": lora_name, "strength_model": float(lora_str)}}
        ltag = f" • 🔥LoRA {lora_name[:20]}(×{lora_str:.2f})" if use_lora else ""
        t0 = time.perf_counter()
        r = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": graph, "client_id": "strata-cell3"}, timeout=30)
        if r.status_code >= 400: _CF_STATE["status"] = f"❌ ComfyUI ปฏิเสธ: {r.text[:300]}"; return
        pid, lp = r.json().get("prompt_id", ""), COMFY_ROOT / "comfyui.log"
        for _ in range(600):
            el = int(time.perf_counter() - t0)
            try: hist = (requests.get(f"http://127.0.0.1:8188/history/{pid}", timeout=8).json() or {}).get(pid) or {}
            except Exception: hist = {}
            if (hist.get("status") or {}).get("status_str") == "error":
                raise RuntimeError(f"ComfyUI error: {json.dumps((hist.get('status') or {}).get('messages', []), ensure_ascii=False)[:350]}")
            imgs = [im for no in (hist.get("outputs") or {}).values() for im in (no.get("images") or []) if im.get("filename")]
            if imgs:
                im0 = imgs[0]; out_dir = Path(_cfg.STRATA_HOME) / "outputs"; out_dir.mkdir(parents=True, exist_ok=True)
                dst = out_dir / f"{key}_{int(time.time())}_{im0['filename']}"
                dst.write_bytes(requests.get("http://127.0.0.1:8188/view", params={"filename": im0['filename'], "subfolder": im0.get("subfolder", ""), "type": im0.get("type", "output")}, timeout=60).content)
                secs = round(time.perf_counter() - t0, 1)
                _CF_STATE["last_img"] = {"uri": _ig.image_data_uri(dst, max_width=1024), "name": dst.name, "seed": seed, "seconds": secs, "lora_info": f"{w}×{h} • CFG {cfg_v}"}
                _CF_STATE["status"] = f"✅ วาดภาพสำเร็จใน {secs}s (Seed: {seed}{ltag})"
                return
            tail = "\n".join(lp.read_text(errors="ignore").splitlines()[-5:]) if lp.is_file() else ""
            if "OutOfMemoryError" in tail or "CUDA out of memory" in tail: _CF_STATE["status"] = f"❌ VRAM ไม่พอ:\n{tail}"; return
            ph = re.findall(r"(\d+%\|.*?\|\s*\d+/\d+)", tail)
            _CF_STATE["status"] = f"🎨 KSampler [{ph[-1] if ph else f'{steps} steps'}] ({w}×{h} • CFG {cfg_v}{ltag})... {el}s"
            time.sleep(1.0)
        _CF_STATE["status"] = "❌ ComfyUI เกิน 10 นาที"
    except Exception as e:
        _CF_STATE["status"] = f"❌ {e}"
    finally:
        _CF_STATE["running"] = False

def _cf_rpc(act: str, payload: dict | None = None) -> dict:
    if act == "queue" and not _CF_STATE["running"]:
        _CF_STATE["running"] = True
        threading.Thread(target=_run_comfy_graph, args=(payload or {},), daemon=True).start()
    return {**_sync_from_cell2(), **_CF_STATE}

if IS_COLAB:
    from google.colab import output as _c_out3  # type: ignore[import-not-found]
    _c_out3.register_callback("strata.comfy_cell", lambda act, payload=None: JSON(_cf_rpc(act, payload)))

_ip3 = get_ipython()
if _ip3 is not None and getattr(_ip3, "kernel", None) is not None:
    def _ct3(comm, _m):
        @comm.on_msg
        def _r(msg):
            b = (msg.get("content") or {}).get("data") or {}
            comm.send({"id": b.get("id", ""), "result": _cf_rpc(str(b.get("act", "poll")), b.get("payload") or {})})
    with contextlib.suppress(Exception): _ip3.kernel.comm_manager.register_target("strata.comfy_cell", _ct3)

_CF_HTML = r"""
<div id="cf-root" style="width:100%;background:#0b1120;color:#f8fafc;border:1px solid #10b981;border-radius:10px;padding:14px 16px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;box-sizing:border-box;">
  <style>
    #cf-root * { box-sizing: border-box; }
    .c-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(265px, 1fr)); gap: 10px; margin-top: 10px; }
    .c-node { background: #111827; border: 1px solid #334155; border-top: 3px solid #10b981; border-radius: 9px; padding: 10px 12px; font-size: 12.5px; }
    .c-node h4 { margin: 0 0 7px 0; font-size: 13px; color: #34d399; display: flex; justify-content: space-between; align-items: center; gap: 4px; }
    .c-in { width: 100%; background: #0f172a; color: #f8fafc; border: 1px solid #475569; border-radius: 6px; padding: 6px 8px; font-size: 12.5px; margin-top: 4px; }
    .c-btn { min-height: 38px; padding: 7px 13px; border-radius: 8px; border: 1px solid #10b981; background: #059669; color: #fff; font-weight: 700; cursor: pointer; font-size: 12.8px; }
    .c-btn:disabled { opacity: 0.42; cursor: not-allowed; filter: grayscale(0.35); }
    .c-chip { padding: 2px 7px; border-radius: 6px; border: 1px solid #475569; background: #1e293b; color: #e2e8f0; font-size: 11px; cursor: pointer; }
    .c-pill { padding: 5px 11px; border-radius: 99px; font-size: 12px; font-weight: 600; border: 1px solid #334155; background: #1e293b; }
  </style>
  <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;padding-bottom:10px;border-bottom:1px solid #1e293b;">
    <div>
      <b style="font-size:15.5px;color:#34d399;">🎨 ComfyUI GGUF Node Workflow Studio (Automation โหนด 4–7 ตามช่องแชทและการรีพายเซลล์ที่ 2)</b>
      <div id="cf-stat" style="font-size:12.5px;color:#fde047;margin-top:3px;"></div>
    </div>
    <div style="display:flex;gap:7px;flex-wrap:wrap;align-items:center;">
      <button type="button" id="cf-autolk" class="c-pill" style="cursor:pointer;background:#2e1065;border-color:#a855f7;color:#e9d5ff;" onclick="cfToggleAuto()">🤖 Auto โหนด 4–7: 🟢 รับค่าสดจากเซลล์ 2</button>
      <span id="cf-sync-pill" class="c-pill"></span>
      <button class="c-btn" id="cf-qbtn" disabled onclick="cfQueue()">🔒 โหลดโมเดลในเซลล์ 2 ก่อน</button>
    </div>
  </div>
  <div class="c-grid">
    <div class="c-node" style="border-top-color:#38bdf8;">
      <h4><span>📦 Node 1–3 & 10: Loaders + LoRA</span><span style="color:#94a3b8;font-size:11px;">AUTO-SYNC</span></h4>
      <div>UNet GGUF: <b id="cf-unet" style="color:#7dd3fc;"></b></div>
      <div style="margin-top:4px;">Text Encoder: <b id="cf-clip" style="color:#c4b5fd;"></b> (<span id="cf-ctype"></span>)</div>
      <div style="margin-top:4px;">VAE: <b id="cf-vae" style="color:#fca5a5;"></b></div>
      <div style="margin-top:9px;padding-top:8px;border-top:1px dashed #334155;">
        <div style="display:flex;justify-content:space-between;align-items:center;"><b style="color:#fbbf24;">🔥 Node 10: LoraLoaderModelOnly</b><span id="cf-lbadge" style="font-size:11px;color:#34d399;"></span></div>
        <select id="cf-lora" class="c-in" onchange="cfUpdLoraBadge()"></select>
        <div style="display:flex;justify-content:space-between;align-items:center;margin-top:6px;"><span>ความแรง LoRA:</span><input id="cf-lstr" type="number" step="0.05" min="-2.0" max="3.0" value="1.00" style="width:70px;padding:3px 6px;background:#0f172a;color:#fde047;font-weight:700;border:1px solid #f59e0b;border-radius:6px;text-align:center;" oninput="cfSyncLora(this.value,'num')"></div>
        <input id="cf-lstr-rng" type="range" min="0" max="2" step="0.05" value="1.0" style="width:100%;margin-top:5px;accent-color:#f59e0b;cursor:pointer;" oninput="cfSyncLora(this.value,'rng')">
        <div style="display:flex;gap:4px;flex-wrap:wrap;margin-top:4px;"><button type="button" class="c-chip" onclick="cfSyncLora('0.00','chip')">0.0</button><button type="button" class="c-chip" onclick="cfSyncLora('0.50','chip')">0.50</button><button type="button" class="c-chip" onclick="cfSyncLora('0.85','chip')">0.85</button><button type="button" class="c-chip" onclick="cfSyncLora('1.00','chip')">1.00</button><button type="button" class="c-chip" onclick="cfSyncLora('1.25','chip')">1.25</button></div>
      </div>
    </div>
    <div class="c-node" style="border-top-color:#a855f7;">
      <h4><span>✍️ Node 4–5: CLIPTextEncode</span><span id="cf-style-tag" style="color:#d8b4fe;font-size:11px;">PROMPT AUTO</span></h4>
      <label>Positive Prompt (ซิงก์และเสริมคำศัพท์อัตโนมัติจากช่องแชท/รีพายเซลล์ 2):</label>
      <textarea id="cf-pos" class="c-in" rows="4">cinematic masterpiece, ultra detailed portrait, dramatic lighting, sharp focus, 8k</textarea>
      <label style="display:block;margin-top:6px;">Negative Prompt (สร้างอัตโนมัติตามโทนภาพ & คำสั่ง <code>ไม่เอา / --no</code>):</label>
      <textarea id="cf-neg" class="c-in" rows="3">blurry, low quality, deformed, bad anatomy, watermark, text</textarea>
    </div>
    <div class="c-node" style="border-top-color:#f59e0b;">
      <h4><span>⚙️ Node 6–7: Latent & KSampler</span><span id="cf-smp-tag" style="color:#fcd34d;font-size:11px;">SAMPLER AUTO</span></h4>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:6px;">
        <label>Width<input id="cf-w" type="number" step="16" value="1024" class="c-in"></label>
        <label>Height<input id="cf-h" type="number" step="16" value="1024" class="c-in"></label>
        <label>Steps<input id="cf-steps" type="number" value="8" class="c-in"></label>
        <label>CFG<input id="cf-cfg" type="number" step="0.5" value="1.0" class="c-in"></label>
        <label>Sampler<input id="cf-smp" value="euler" class="c-in"></label>
        <label>Scheduler<input id="cf-sch" value="simple" class="c-in"></label>
      </div>
      <label style="display:block;margin-top:5px;">Seed (-1 = สุ่มใหม่ทุกครั้ง หรือ <code>--seed N</code> ในช่องแชท)<input id="cf-seed" type="number" value="-1" class="c-in"></label>
    </div>
    <div class="c-node" style="border-top-color:#10b981;">
      <h4><span>🖼️ Node 8–9: VAEDecode → SaveImage</span><span id="cf-meta" style="color:#94a3b8;font-size:11px;">OUTPUT</span></h4>
      <div id="cf-out" style="min-height:225px;display:flex;align-items:center;justify-content:center;background:#0f172a;border:1px dashed #334155;border-radius:8px;padding:8px;text-align:center;color:#94a3b8;">พิมพ์หรือรีพายข้อความในช่องแชทเซลล์ที่ 2 แล้วโหนด 4–7 จะปรับค่าเข้าชุดให้อัตโนมัติ จากนั้นกด ▶️ Queue Prompt ด้านบนได้เลย</div>
    </div>
  </div>
</div>
<script>
(function(){
  let C = __CF_INIT__; let shown = ''; let lastModelSig = ''; let lastWfTs = 0; let autoNodes = true;
  let _cm = null; const _wt = {};
  function esc(x){ const d=document.createElement('div'); d.textContent=String(x||''); return d.innerHTML; }
  function fitH(){ try{ if(typeof google!=='undefined'&&google.colab&&google.colab.output&&google.colab.output.setIframeHeight) google.colab.output.setIframeHeight(Math.max(500, document.getElementById('cf-root').scrollHeight+20), true, {maxHeight: 3000}); }catch(_){} }
  async function rpc(a,p){
    if(typeof google!=='undefined'&&google.colab&&google.colab.kernel){
      const r = await google.colab.kernel.invokeFunction('strata.comfy_cell',[a,p||{}],{});
      return (r&&r.data&&r.data['application/json'])?r.data['application/json']:r;
    }
    if(typeof Jupyter!=='undefined'&&Jupyter.notebook&&Jupyter.notebook.kernel){
      if(!_cm){ _cm=Jupyter.notebook.kernel.comm_manager.new_comm('strata.comfy_cell',{}); _cm.on_msg(m=>{const d=(m.content||{}).data||{}; if(_wt[d.id]){_wt[d.id](d.result);delete _wt[d.id];}}); }
      return new Promise(res => { const id = String(Date.now()) + Math.random(); _wt[id] = res; _cm.send({id, act: a, payload: p||{}}); });
    }
    return null;
  }
  window.cfToggleAuto = function(){
    autoNodes = !autoNodes;
    const bt = document.getElementById('cf-autolk');
    bt.textContent = autoNodes ? '🤖 Auto โหนด 4–7: 🟢 รับค่าสดจากเซลล์ 2' : '✋ Auto โหนด 4–7: ⏸️ ล็อกปรับค่าเอง';
    bt.style.background = autoNodes ? '#2e1065' : '#1e293b';
    if(autoNodes){ lastWfTs = 0; render(); }
  };
  window.cfUpdLoraBadge = function(){
    const fn = document.getElementById('cf-lora').value; const v = parseFloat(document.getElementById('cf-lstr').value || '1.0');
    const b = document.getElementById('cf-lbadge');
    if(!fn || Math.abs(v) < 1e-6){ b.style.color = '#94a3b8'; b.textContent = '⏸️ Bypass'; }
    else { b.style.color = '#34d399'; b.textContent = `🟢 Active ×${v.toFixed(2)}`; }
  };
  window.cfSyncLora = function(val, src){
    const num = parseFloat(val); if(isNaN(num)) return;
    if(src !== 'num') document.getElementById('cf-lstr').value = num.toFixed(2);
    if(src !== 'rng') document.getElementById('cf-lstr-rng').value = String(Math.max(0, Math.min(2, num)));
    cfUpdLoraBadge();
  };
  function applyAutoWf(wf){
    if(!wf || !autoNodes) return;
    document.getElementById('cf-pos').value = wf.pos || '';
    document.getElementById('cf-neg').value = wf.neg || '';
    document.getElementById('cf-w').value = wf.width || 1024;
    document.getElementById('cf-h').value = wf.height || 1024;
    document.getElementById('cf-steps').value = wf.steps || 8;
    document.getElementById('cf-cfg').value = wf.cfg !== undefined ? wf.cfg : 1.0;
    document.getElementById('cf-smp').value = wf.sampler || 'euler';
    document.getElementById('cf-sch').value = wf.scheduler || 'simple';
    document.getElementById('cf-seed').value = wf.seed !== undefined ? wf.seed : -1;
    document.getElementById('cf-style-tag').textContent = wf.style_label || 'AUTO';
    document.getElementById('cf-smp-tag').textContent = `${wf.width}×${wf.height} • ${wf.steps}st • CFG ${wf.cfg}`;
  }
  function render(){
    document.getElementById('cf-stat').textContent = C.status || '';
    const p = C.active || {};
    const pill = document.getElementById('cf-sync-pill'); const qbtn = document.getElementById('cf-qbtn');
    if(p.ready && p.key){ pill.style.background = '#064e3b'; pill.style.borderColor = '#10b981'; pill.style.color = '#6ee7b7'; pill.textContent = '🔗 โมเดล: ✅ ' + p.key; }
    else { pill.style.background = '#450a0a'; pill.style.borderColor = '#ef4444'; pill.style.color = '#fca5a5'; pill.textContent = '🔒 รอโหลดโมเดลภาพในเซลล์ 2'; }
    if(C.running){ qbtn.disabled = true; qbtn.textContent = '⏳ กำลังวาดภาพ...'; }
    else if(!p.ready || !p.key){ qbtn.disabled = true; qbtn.textContent = '🔒 โหลดโมเดลในเซลล์ 2 ก่อน'; }
    else { qbtn.disabled = false; qbtn.textContent = '▶️ Queue Prompt (วาดภาพ)'; }
    const lsel = document.getElementById('cf-lora'); const prevLora = lsel.value;
    lsel.innerHTML = '<option value="">— ไม่ใช้ LoRA (Bypass โหนด 10) —</option>' + (C.loras||[]).map(fn=>`<option value="${esc(fn)}">🔥 ${esc(fn)}</option>`).join('');
    const mSig = `${p.key}|${p.ready?1:0}|${C.synced_lora}|${C.synced_lora_strength}|${(C.loras||[]).join(',')}`;
    if(mSig !== lastModelSig){
      lastModelSig = mSig;
      document.getElementById('cf-unet').textContent = p.unet || '-';
      document.getElementById('cf-clip').textContent = p.clip || '-';
      document.getElementById('cf-ctype').textContent = p.clip_type || '-';
      document.getElementById('cf-vae').textContent = p.vae || '-';
      if(!C.workflow && autoNodes){
        document.getElementById('cf-steps').value = p.steps || 8;
        document.getElementById('cf-cfg').value = p.cfg !== undefined ? p.cfg : 1.0;
        document.getElementById('cf-smp').value = p.sampler || 'euler';
        document.getElementById('cf-sch').value = p.scheduler || 'simple';
      }
      lsel.value = (C.synced_lora && [...lsel.options].some(o=>o.value===C.synced_lora)) ? C.synced_lora : '';
      cfSyncLora(String(C.synced_lora_strength !== undefined ? C.synced_lora_strength : 1.0), 'chip');
    } else {
      if(prevLora && [...lsel.options].some(o=>o.value===prevLora)) lsel.value = prevLora;
      cfUpdLoraBadge();
    }
    if(C.workflow && C.workflow.wf_ts && C.workflow.wf_ts !== lastWfTs){ lastWfTs = C.workflow.wf_ts; applyAutoWf(C.workflow); }
    if(C.last_img && C.last_img.name !== shown){
      shown = C.last_img.name;
      document.getElementById('cf-meta').textContent = `${C.last_img.seconds}s • Seed ${C.last_img.seed} • ${C.last_img.lora_info||''}`;
      document.getElementById('cf-out').innerHTML = `<div><img src="${C.last_img.uri}" style="max-width:100%;border-radius:7px;display:block;margin-bottom:6px;"/><a href="${C.last_img.uri}" download="${esc(C.last_img.name)}" class="c-btn" style="display:inline-block;text-decoration:none;font-size:12px;padding:5px 10px;">💾 บันทึกรูป (${esc(C.last_img.name)})</a></div>`;
    }
    fitH();
  }
  window.cfQueue = async function(){
    if(!C.active || !C.active.ready){ render(); return; }
    const r = await rpc('queue',{
      lora_name: document.getElementById('cf-lora').value, lora_strength: parseFloat(document.getElementById('cf-lstr').value||'1.0'),
      pos: document.getElementById('cf-pos').value, neg: document.getElementById('cf-neg').value,
      width: parseInt(document.getElementById('cf-w').value||'1024',10), height: parseInt(document.getElementById('cf-h').value||'1024',10),
      steps: parseInt(document.getElementById('cf-steps').value||'8',10), cfg: parseFloat(document.getElementById('cf-cfg').value||'1.0'),
      sampler: document.getElementById('cf-smp').value, scheduler: document.getElementById('cf-sch').value,
      seed: parseInt(document.getElementById('cf-seed').value||'-1',10)
    });
    if(r){ C=r; render(); }
  };
  render();
  let pollBusy = false;
  setInterval(async ()=>{
    if(pollBusy) return;
    pollBusy = true;
    try {
      const r = await rpc('poll', {});
      if(r){ C = r; render(); }
    } catch(_){} finally {
      pollBusy = false;
    }
  }, 1800);
})();
</script>
"""
display(HTML(_CF_HTML.replace("__CF_INIT__", json.dumps(_cf_rpc("poll"), ensure_ascii=False).replace("</", "<\\\\/"))))
